# Mission 3 ver29 — KLUE-ASL + Qwen3-Embedding / Modal L4

**기존 `mission3-ver29-data` Volume과 L4 ×1을 이 노트북에 연결한 뒤 시작하세요.**
GPU와 Volume 설정은 새 노트북에서 다시 확인합니다. 기존 입력 ZIP을 그대로 사용합니다.

1. Files에서 `/mnt/mission3-ver29-data`에 `mission3_ver29_inputs.zip`을 업로드합니다.
   이미 있다면 다시 올리지 않아도 됩니다. 압축은 직접 풀지 않습니다.
2. 처음에는 아래 기본값 `screening`을 유지하고 **Run All을 한 번** 누릅니다.
3. 두 후보를 순서대로 학습합니다. L4의 계산 정밀도와 빠른 배치 크기는 자동 선택합니다.
   예상 시간은 실제 배치 속도 측정 후 출력합니다.
4. 종료 후 Volume의 `exports/ver29_analysis_*.zip`을 내려받습니다.
   전체 모델과 재개용 checkpoint는 `results` 폴더에 남겨 두세요.
5. 저장된 파일을 확인한 뒤 **Stop kernel**로 GPU를 종료합니다.

Training 내부 fit 20,440개로 KLUE-ASL과 Qwen3-Embedding-0.6B를 각각 최대 3 epoch 학습합니다.
고정된 기존 ver28, 새 단독 모델 둘, 새 모델 1:1 평균, 네 모델 동일 평균을 development에서 비교하고
선택을 고정한 뒤 confirmation을 확인합니다. **전체 Training 학습은 자동 시작하지 않습니다.**
데이터 분할·학습 방식·선택 정책은 기존 ver29와 같습니다. F1 0.7 초과 여부는 실제 결과로 확인합니다.

한 번에 노트북 하나만 학습하세요. 이미 실행 중이면 Run All을 다시 누르지 마세요.
시간 예산은 설치부터 저장까지 최대 480분입니다. 부족하면 `paused`로 저장하고 나중에 재개합니다.
무료 크레딧 소진이나 Modal 강제 종료까지 코드가 보장하지는 않습니다.
첫 실행은 새 screening으로 시작하세요. Kaggle T4의 FP16 중간 상태를 L4 BF16로 바로 재개하지 않습니다.


In [ ]:
# Edit only these fields. Keep defaults for the first L4 run.
VOLUME_DIR = "/mnt/mission3-ver29-data"
RUN_MODE = "screening"  # "screening" or "full"
SOURCE_RUN = ""  # Full only: exact completed screening folder on the Volume
RESUME_RUN = ""  # Resume only: exact paused folder from the same phase and GPU precision
INPUT_ZIP = ""  # Empty: VOLUME_DIR/mission3_ver29_inputs.zip
TIME_BUDGET_MINUTES = 480  # Installation, downloads, evaluation and saving are included

import time
if type(TIME_BUDGET_MINUTES) is not int or not 30 <= TIME_BUDGET_MINUTES <= 480:
    raise ValueError("TIME_BUDGET_MINUTES must be an integer between 30 and 480")
INVOCATION_STARTED = time.monotonic()
DEADLINE_MONOTONIC = INVOCATION_STARTED + TIME_BUDGET_MINUTES * 60
PREPARED = False
DEPENDENCIES_PREPARED = False
LAUNCHED = False
print("실행 모드:", RUN_MODE, "· 최대", TIME_BUDGET_MINUTES, "분")
print("영구 저장 Volume:", VOLUME_DIR)


In [ ]:
# Restore exact ver29 sources and verify the supplied input before installing packages.
import importlib.util
import json
import os
from pathlib import Path
import sys
import tempfile
import uuid

RUNTIME_SOURCES = {'__init__.py': '"""Mission 3 ver29: Kaggle supervised encoder comparison and separate refit."""\n', 'analysis.py': '"""Fixed-decision multilabel scores and paired call-level uncertainty."""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom sklearn.metrics import average_precision_score\n\nfrom .data import TARGET_SYMPTOMS, _binary, score_predictions\n\n\ndef _logits(values, rows=None):\n    values = np.asarray(values, dtype=np.float64)\n    if (values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS)\n            or (rows is not None and len(values) != rows) or not np.isfinite(values).all()):\n        raise ValueError("logits must be a finite (samples, 9) array aligned with targets")\n    return values\n\n\ndef metrics(targets, logits):\n    """Every logit >= 0 is positive, including exact zero ties."""\n    targets = _binary(targets, name="targets")\n    logits = _logits(logits, len(targets))\n    predicted = logits >= 0\n    result = score_predictions(targets, predicted)\n    ap_values = []\n    for index, symptom in enumerate(TARGET_SYMPTOMS):\n        truth, prediction = targets[:, index].astype(bool), predicted[:, index]\n        row = result["per_symptom"][symptom]\n        # AP is undefined without positive targets; JSON null makes this explicit.\n        ap = float(average_precision_score(truth, logits[:, index])) if truth.any() else None\n        row.update({"true_positive": int((truth & prediction).sum()),\n                    "true_negative": int((~truth & ~prediction).sum()), "average_precision": ap})\n        if ap is not None:\n            ap_values.append(ap)\n    result["macro_average_precision"] = float(np.mean(ap_values)) if ap_values else None\n    result["average_precision_valid_labels"] = len(ap_values)\n    result["nausea_f1"] = result["per_symptom"]["오심"]["f1"]\n    result["decision_rule"] = "independent logits >= 0 (sigmoid >= 0.5); zero ties are positive"\n    return result\n\n\ndef ensemble_logits(control_logits, candidate_logits):\n    """Return logit of the fixed 1:1 sigmoid probability mean.\n\n    Compute the probability mean in log space, preserving its ranking for AP.\n    Its mathematical decision is a+b >= 0. Roundoff in the log-space\n    subtraction is repaired with that identity, without a fitted offset.\n    Same-sign inputs use their signs directly; opposite-sign addition cannot\n    overflow and retains the decision even for subnormal floating-point inputs.\n    """\n    left = _logits(control_logits)\n    right = _logits(candidate_logits, len(left))\n    log_positive = np.logaddexp(-np.logaddexp(0., -left), -np.logaddexp(0., -right))\n    log_negative = np.logaddexp(-np.logaddexp(0., left), -np.logaddexp(0., right))\n    mixed = log_positive - log_negative\n    same_sign = np.signbit(left) == np.signbit(right)\n    decision = np.sign(left) + np.sign(right)\n    decision[~same_sign] = left[~same_sign] + right[~same_sign]\n    nonzero = decision != 0\n    mixed[nonzero] = np.copysign(np.maximum(np.abs(mixed[nonzero]), np.nextafter(0., 1.)), decision[nonzero])\n    mixed[~nonzero] = 0.\n    return mixed\n\n\ndef equal_mean_logits(members):\n    """Fixed equal probability mean for the registered 1/2/4-member recipes.\n\n    Four members use two equal pairs and an equal pair of pairs. This is\n    algebraically the equal four-model probability mean and matches the\n    preserved ver28 baseline pair during screening and offline inference.\n    """\n    values = list(members)\n    if len(values) not in (1, 2, 4):\n        raise ValueError("Only the registered one, two, or four model mean is supported")\n    if len(values) == 1:\n        return _logits(values[0]).copy()\n    if len(values) == 2:\n        return ensemble_logits(*values)\n    return ensemble_logits(ensemble_logits(*values[:2]), ensemble_logits(*values[2:]))\n\n\ndef paired_error_changes(targets, control_logits, candidate_logits):\n    """Compare paired rows; common errors identify remaining complementarity."""\n    targets = _binary(targets, name="targets").astype(bool)\n    control = _logits(control_logits, len(targets)) >= 0\n    candidate = _logits(candidate_logits, len(targets)) >= 0\n    if not len(targets):\n        raise ValueError("Cannot compare an empty dataset")\n    result = {"samples": len(targets), "per_symptom": {}}\n    for index, symptom in enumerate(TARGET_SYMPTOMS):\n        truth, old, new = targets[:, index], control[:, index], candidate[:, index]\n        result["per_symptom"][symptom] = {\n            "common_false_negative": int((truth & ~old & ~new).sum()),\n            "common_false_positive": int((~truth & old & new).sum()),\n            "fixed_false_negative": int((truth & ~old & new).sum()),\n            "fixed_false_positive": int((~truth & old & ~new).sum()),\n            "new_false_negative": int((truth & old & ~new).sum()),\n            "new_false_positive": int((~truth & ~old & new).sum()),\n            "control_false_negative": int((truth & ~old).sum()),\n            "control_false_positive": int((~truth & old).sum()),\n            "candidate_false_negative": int((truth & ~new).sum()),\n            "candidate_false_positive": int((~truth & new).sum()),\n        }\n    return result\n\n\ndef _f1_from_counts(tp, fp, fn):\n    denominator = 2 * tp + fp + fn\n    return np.divide(2 * tp, denominator, out=np.zeros_like(tp, dtype=np.float64), where=denominator != 0)\n\n\ndef paired_bootstrap_ci(targets, control_logits, candidate_logits, iterations: int = 500,\n                        seed: int = 42, confidence: float = .95):\n    """Percentile CIs for candidate-minus-control F1, resampling entire calls.\n\n    Both models and all nine labels use the identical resampled row indices.\n    This estimates uncertainty conditional on these trained models and split;\n    it does not include training seed or repeated selection uncertainty.\n    """\n    truth = _binary(targets, name="targets").astype(bool)\n    old = _logits(control_logits, len(truth)) >= 0\n    new = _logits(candidate_logits, len(truth)) >= 0\n    if not len(truth):\n        raise ValueError("Cannot bootstrap an empty dataset")\n    if type(iterations) is not int or iterations < 2:\n        raise ValueError("iterations must be an integer >= 2")\n    if type(seed) is not int or seed < 0 or not 0 < confidence < 1:\n        raise ValueError("seed must be nonnegative and confidence must be in (0, 1)")\n    # One row contains every class and both predictions: a call is the unit.\n    counts = np.stack([np.stack([truth & prediction, ~truth & prediction, truth & ~prediction], axis=1)\n                       for prediction in (old, new)], axis=1).astype(np.int8)\n    point = _f1_from_counts(*counts.sum(axis=0).transpose(1, 0, 2))\n    point_delta = point[1] - point[0]\n    rng = np.random.default_rng(seed)\n    delta = np.empty((iterations, len(TARGET_SYMPTOMS)), dtype=np.float64)\n    for iteration in range(iterations):\n        sampled = counts[rng.integers(0, len(truth), size=len(truth))].sum(axis=0)\n        f1 = _f1_from_counts(*sampled.transpose(1, 0, 2))\n        delta[iteration] = f1[1] - f1[0]\n    quantiles = [(1 - confidence) / 2, (1 + confidence) / 2]\n\n    def interval(values, estimate):\n        lower, upper = np.quantile(values, quantiles)\n        return {"estimate": float(estimate), "lower": float(lower), "upper": float(upper)}\n\n    return {"samples": len(truth), "iterations": iterations, "seed": seed, "confidence": confidence,\n            "unit": "call", "comparison": "candidate minus control", "method": "paired percentile bootstrap",\n            "macro_f1_delta": interval(delta.mean(axis=1), point_delta.mean()),\n            "per_symptom_f1_delta": {name: interval(delta[:, index], point_delta[index])\n                                     for index, name in enumerate(TARGET_SYMPTOMS)},\n            "limitations": "Conditional on trained models and this split; repeated selection and training-seed "\n                           "uncertainty are not included. Duplicate calls may still be statistically dependent."}\n', 'bootstrap.py': '"""Safely restore label JSONs only; old model weights are never extracted."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport stat\nimport tempfile\nimport zipfile\n\nTRAIN_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급/"\nVALID_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급/"\nMARKER_NAME = ".mission3_label_bundle_complete.json"\n\n\ndef _relative(name):\n    path = PurePosixPath(name)\n    if (not isinstance(name, str) or not name or "\\\\" in name or ":" in name\n            or path.is_absolute() or ".." in path.parts\n            or path.as_posix() != name.rstrip("/")):\n        raise ValueError(f"Unsafe bundle path: {name}")\n    return Path(*path.parts)\n\n\ndef _target(root, name):\n    target = root / _relative(name)\n    if not target.resolve().is_relative_to(root):\n        raise ValueError(f"Bundle path escapes extraction directory: {name}")\n    # Existing links must not redirect even to another location inside the root.\n    for component in (target, *target.parents):\n        if component == root.parent:\n            break\n        if component.is_symlink():\n            raise ValueError(f"Extraction destination contains a symlink: {name}")\n    return target\n\n\ndef _hash(path):\n    digest = hashlib.sha256()\n    with path.open("rb") as reader:\n        for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _selected_files(archive):\n    infos = archive.infolist()\n    names = set()\n    entry_types = {}\n    for info in infos:\n        # Windows ZipInfo normalizes backslashes in filename; orig_filename\n        # retains the stored member spelling that must also pass validation.\n        _relative(info.orig_filename)\n        _relative(info.filename)\n        normalized = info.filename.rstrip("/").casefold()\n        if normalized in names:\n            raise ValueError(f"Duplicate ZIP member or directory: {info.filename}")\n        names.add(normalized)\n        entry_types[normalized] = info.is_dir()\n        if stat.S_ISLNK(info.external_attr >> 16) or info.flag_bits & 1:\n            raise ValueError("Links and encrypted ZIP entries are not supported")\n    for name in entry_types:\n        if any(parent.as_posix() in entry_types and not entry_types[parent.as_posix()]\n               for parent in PurePosixPath(name).parents if parent.as_posix() != "."):\n            raise ValueError("ZIP file/directory collision")\n    selected = [info for info in infos if not info.is_dir() and info.filename.endswith(".json")\n                and info.filename.startswith((TRAIN_PREFIX, VALID_PREFIX))]\n    for prefix in (TRAIN_PREFIX, VALID_PREFIX):\n        subset = [info for info in selected if info.filename.startswith(prefix)]\n        if not subset:\n            raise ValueError(f"Input ZIP has no label JSONs in {prefix}")\n        if len({PurePosixPath(info.filename).name.casefold() for info in subset}) != len(subset):\n            raise ValueError("Duplicate label JSON filenames in input ZIP")\n    by_name = {info.filename: info for info in selected}\n    if "bundle_manifest.json" in archive.namelist():\n        raw = archive.read("bundle_manifest.json")\n        manifest = json.loads(raw)\n        if not isinstance(manifest, dict) or manifest.get("format") != "mission3_ver5_inputs_v1":\n            raise ValueError("Unsupported Mission 3 input bundle manifest")\n        files = manifest.get("files")\n        if not isinstance(files, list):\n            raise ValueError("Bundle manifest has no files")\n        metadata, listed = {}, set()\n        for row in files:\n            if not isinstance(row, dict) or not isinstance(row.get("path"), str):\n                raise ValueError("Invalid bundle manifest file entry")\n            name = row["path"]\n            _relative(name)\n            if name.casefold() in listed:\n                raise ValueError(f"Duplicate manifest entry: {name}")\n            listed.add(name.casefold())\n            if name not in by_name:\n                continue\n            size, digest = row.get("size"), row.get("sha256")\n            if (type(size) is not int or size < 0 or not isinstance(digest, str)\n                    or re.fullmatch(r"[0-9a-f]{64}", digest) is None):\n                raise ValueError(f"Invalid manifest metadata: {name}")\n            if by_name[name].file_size != size:\n                raise ValueError(f"Bundle size mismatch: {name}")\n            metadata[name] = {"path": name, "size": size, "sha256": digest}\n        if set(metadata) != set(by_name):\n            raise ValueError("Bundle manifest does not list every label JSON")\n        for prefix, key in ((TRAIN_PREFIX, "training_count"), (VALID_PREFIX, "validation_count")):\n            count = sum(name.startswith(prefix) for name in metadata)\n            if type(manifest.get(key)) is not int or manifest[key] != count:\n                raise ValueError(f"Bundle JSON count mismatch: {key}")\n        signature = hashlib.sha256(raw).hexdigest()\n    else:\n        # Without a manifest, hash only label contents, never the large old models.\n        metadata = {}\n        for name, info in sorted(by_name.items()):\n            digest, size = hashlib.sha256(), 0\n            with archive.open(info) as reader:\n                for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n                    digest.update(chunk)\n                    size += len(chunk)\n            if size != info.file_size:\n                raise ValueError(f"Bundle size mismatch: {name}")\n            metadata[name] = {"path": name, "size": size, "sha256": digest.hexdigest()}\n        signature = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()\n    return [metadata[name] for name in sorted(metadata)], signature\n\n\ndef _validate_directory(directory, name):\n    directory = Path(directory).expanduser().resolve()\n    if not directory.is_dir() or not any(path.is_file() for path in directory.rglob("*.json")):\n        raise ValueError(f"{name} label directory does not contain JSON files: {directory}")\n    return directory\n\n\ndef prepare_data(input_zip, work_dir, train_dir="", validation_dir=""):\n    """Use two explicit label directories, or verify/extract the existing bundle.\n\n    Reuse requires a completion marker plus fresh content hashes. This detects\n    changed files even when their size and modification time are preserved.\n    """\n    if bool(train_dir) != bool(validation_dir):\n        raise ValueError("Set both train_dir and validation_dir, or leave both empty")\n    if train_dir:\n        train = _validate_directory(train_dir, "Training")\n        valid = _validate_directory(validation_dir, "Validation")\n        if train == valid or train.is_relative_to(valid) or valid.is_relative_to(train):\n            raise ValueError("Training and Validation label directories must be distinct and disjoint")\n        return train, valid\n    archive_path = Path(input_zip).expanduser().resolve()\n    if not archive_path.is_file():\n        raise FileNotFoundError(f"Input ZIP is missing: {archive_path}")\n    requested_root = Path(work_dir).expanduser()\n    if requested_root.is_symlink():\n        raise ValueError("Extraction work_dir must not be a symlink")\n    root = requested_root.resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    marker = root / MARKER_NAME\n    with zipfile.ZipFile(archive_path) as archive:\n        files, signature = _selected_files(archive)\n        # Resolve every destination before any extraction, including existing links.\n        destinations = {row["path"]: _target(root, row["path"]) for row in files}\n        expected_paths = set(destinations.values())\n        for prefix in (TRAIN_PREFIX, VALID_PREFIX):\n            directory = root / prefix\n            if directory.exists():\n                unexpected = set(directory.rglob("*.json")) - expected_paths\n                if unexpected:\n                    raise ValueError("Extraction directory contains label JSONs from another input bundle")\n        previous = None\n        if marker.exists():\n            if marker.is_symlink():\n                raise ValueError("Completion marker must not be a symlink")\n            try:\n                previous = json.loads(marker.read_text(encoding="utf-8"))\n            except (OSError, ValueError):\n                previous = None\n        expected = {row["path"]: {"size": row["size"], "sha256": row["sha256"]} for row in files}\n        reuse = (isinstance(previous, dict) and previous.get("signature") == signature\n                 and previous.get("files") == expected)\n        pending = []\n        for row in files:\n            target = destinations[row["path"]]\n            verified = (target.is_file() and target.stat().st_size == row["size"]\n                        and _hash(target) == row["sha256"])\n            if not verified:\n                pending.append(row)\n        if reuse and not pending:\n            return root / TRAIN_PREFIX, root / VALID_PREFIX\n        if marker.exists():\n            marker.unlink()\n        if shutil.disk_usage(root).free < sum(row["size"] for row in pending):\n            raise OSError("Insufficient local disk space to extract label JSONs")\n        for row in pending:\n            target = destinations[row["path"]]\n            target.parent.mkdir(parents=True, exist_ok=True)\n            temporary = None\n            try:\n                digest, size = hashlib.sha256(), 0\n                with archive.open(row["path"]) as reader, tempfile.NamedTemporaryFile(\n                        dir=target.parent, prefix=".extract_", suffix=".tmp", delete=False) as writer:\n                    temporary = Path(writer.name)\n                    for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n                        writer.write(chunk)\n                        digest.update(chunk)\n                        size += len(chunk)\n                if size != row["size"] or digest.hexdigest() != row["sha256"]:\n                    raise ValueError(f"ZIP content verification failed: {row[\'path\']}")\n                temporary.replace(target)\n            finally:\n                if temporary is not None:\n                    temporary.unlink(missing_ok=True)\n        # Do not publish an incomplete extraction as a reusable data set.\n        temporary = None\n        try:\n            with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=root,\n                                             prefix=".marker_", suffix=".tmp", delete=False) as writer:\n                temporary = Path(writer.name)\n                json.dump({"signature": signature, "files": expected}, writer, ensure_ascii=False)\n            temporary.replace(marker)\n        finally:\n            if temporary is not None:\n                temporary.unlink(missing_ok=True)\n    return root / TRAIN_PREFIX, root / VALID_PREFIX\n', 'budget.py': '"""Share a wall-clock deadline across setup, training, and final export.\n\nCreate the monotonic deadline once per notebook invocation, before package\ninstallation. Do not restore a monotonic timestamp from an earlier Colab VM.\n``Budget`` reserves finalization time when ``training=True``; subprocess\nsupervision enforces the actual invocation deadline. A ``None`` deadline\ndisables time limits while retaining normal completion and error handling.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom numbers import Real\nimport os\nimport signal\nimport subprocess\nimport time\n\n\nclass BudgetExceeded(TimeoutError):\n    """Work stopped without declaring an incomplete artifact complete."""\n\n    def __init__(self, stage, *, deadline_monotonic=None):\n        self.stage = str(stage)\n        self.deadline_monotonic = deadline_monotonic\n        super().__init__(f"Wall-clock budget exhausted during {self.stage}")\n\n\ndef _finite_nonnegative(value, name):\n    if isinstance(value, bool) or not isinstance(value, Real):\n        raise ValueError(f"{name} must be a finite nonnegative number")\n    value = float(value)\n    if not math.isfinite(value) or value < 0:\n        raise ValueError(f"{name} must be a finite nonnegative number")\n    return value\n\n\nclass Budget:\n    """A deadline with conservative admission checks for expensive stages.\n\n    ``remaining(training=True)`` subtracts the export/evaluation reserve.\n    ``check`` raises at an exhausted deadline and otherwise returns seconds left.\n    ``can_start`` applies the safety factor to an estimated duration; it does not\n    extend the deadline or admit work when no time remains.\n    A ``None`` deadline admits work without a time limit and reports ``None``\n    remaining, so saved reports stay valid JSON without infinity values.\n    """\n\n    def __init__(self, deadline_monotonic, finalization_reserve_s=1200,\n                 safety_factor=1.25, clock=time.monotonic):\n        self.deadline_monotonic = (None if deadline_monotonic is None else\n                                   _finite_nonnegative(deadline_monotonic, "deadline_monotonic"))\n        self.finalization_reserve_s = _finite_nonnegative(finalization_reserve_s, "finalization_reserve_s")\n        self.safety_factor = _finite_nonnegative(safety_factor, "safety_factor")\n        if self.safety_factor < 1:\n            raise ValueError("safety_factor must be at least one")\n        if not callable(clock):\n            raise ValueError("clock must be callable")\n        self.clock = clock\n\n    def remaining(self, training=False):\n        if not isinstance(training, bool):\n            raise ValueError("training must be a boolean")\n        if self.deadline_monotonic is None:\n            return None\n        now = _finite_nonnegative(self.clock(), "clock result")\n        reserve = self.finalization_reserve_s if training else 0\n        return max(0.0, self.deadline_monotonic - reserve - now)\n\n    def check(self, stage, training=False):\n        remaining = self.remaining(training=training)\n        if remaining is not None and remaining <= 0:\n            raise BudgetExceeded(stage, deadline_monotonic=self.deadline_monotonic)\n        return remaining\n\n    def can_start(self, estimated_seconds, training=True):\n        estimate = _finite_nonnegative(estimated_seconds, "estimated_seconds")\n        remaining = self.remaining(training=training)\n        return remaining is None or (remaining > 0 and estimate * self.safety_factor <= remaining)\n\n\ndef _signal_process(process, force=False):\n    """Signal only this supervisor\'s new process group (or Windows child)."""\n    try:\n        if os.name == "posix":\n            # Popen(start_new_session=True) makes this child\'s PID its group ID.\n            # Signal the group even if its leader exited, to stop descendants.\n            os.killpg(process.pid, signal.SIGKILL if force else signal.SIGTERM)\n        elif process.poll() is None:\n            process.kill() if force else process.terminate()\n    except ProcessLookupError:\n        pass\n\n\ndef _stop_process(process, deadline_monotonic):\n    """Use at most the remaining shutdown margin for graceful stop and reaping."""\n    _signal_process(process)\n    remaining = max(0.0, deadline_monotonic - time.monotonic())\n    try:\n        process.wait(timeout=min(5.0, remaining / 2))\n    except subprocess.TimeoutExpired:\n        pass\n    # A descendant may ignore SIGTERM even when the group leader has exited.\n    _signal_process(process, force=True)\n    try:\n        process.wait(timeout=max(0.0, deadline_monotonic - time.monotonic()))\n    except subprocess.TimeoutExpired:\n        # SIGKILL/TerminateProcess has already been issued; never wait unbounded.\n        pass\n\n\ndef run_supervised(command, deadline_monotonic, shutdown_margin_s=10, cwd=None, env=None):\n    """Run a command with inherited output and an optional wall-clock watchdog.\n\n    Returns zero on success. Nonzero exit raises ``CalledProcessError``; timeout\n    raises ``BudgetExceeded`` after terminating the child. POSIX descendants are\n    stopped as a group; Windows supervision stops only its own direct child.\n    The shutdown margin is inside the supplied deadline, not added after it.\n    ``None`` waits for completion without a watchdog; error cleanup stays bounded.\n    Commands must be argument sequences and are never executed through a shell.\n    """\n    deadline = (None if deadline_monotonic is None else\n                _finite_nonnegative(deadline_monotonic, "deadline_monotonic"))\n    margin = _finite_nonnegative(shutdown_margin_s, "shutdown_margin_s")\n    if margin <= 0:\n        raise ValueError("shutdown_margin_s must be positive")\n    if isinstance(command, (str, bytes)) or not isinstance(command, (list, tuple)) or not command:\n        raise ValueError("command must be a nonempty argument sequence")\n    if deadline is not None and deadline - margin - time.monotonic() <= 0:\n        raise BudgetExceeded("subprocess startup", deadline_monotonic=deadline)\n    process = subprocess.Popen(command, cwd=cwd, env=env, start_new_session=os.name == "posix")\n    try:\n        timeout = None if deadline is None else max(0.0, deadline - margin - time.monotonic())\n        returncode = process.wait(timeout=timeout)\n    except subprocess.TimeoutExpired:\n        _stop_process(process, deadline if deadline is not None else time.monotonic() + margin)\n        if deadline is None:\n            raise\n        raise BudgetExceeded("subprocess execution", deadline_monotonic=deadline) from None\n    except BaseException:\n        cleanup_deadline = time.monotonic() + margin\n        _stop_process(process, cleanup_deadline if deadline is None else min(deadline, cleanup_deadline))\n        raise\n    if returncode:\n        cleanup_deadline = time.monotonic() + margin\n        _stop_process(process, cleanup_deadline if deadline is None else min(deadline, cleanup_deadline))\n        raise subprocess.CalledProcessError(returncode, command)\n    return returncode\n', 'build_notebook.py': '"""Build a standalone Kaggle notebook containing the ver29 runtime sources."""\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nimport textwrap\n\nHERE = Path(__file__).resolve().parent\n\n\ndef source_files():\n    """Include all runtime modules without embedding tests or generated outputs."""\n    modules = sorted(path.name for path in HERE.glob("*.py")\n                     if not path.name.startswith("test_") and path.name != "conftest.py")\n    return modules + ["model_specs.json", "requirements.txt", "README.md"]\n\n\ndef make_notebook():\n    sources = {name: (HERE / name).read_text(encoding="utf-8") for name in source_files()}\n    required = {"__init__.py", "settings.py", "run_notebook.py", "runtime.py", "budget.py", "trainer.py",\n                "reference.py"}\n    if not required.issubset(sources):\n        raise FileNotFoundError("Incomplete ver29 runtime source package")\n    for name, source in sources.items():\n        if name.endswith(".py"):\n            compile(source, name, "exec")\n    cells = []\n\n    def add(kind, identifier, source):\n        cell = {"cell_type": kind, "id": "ver29-" + identifier, "metadata": {},\n                "source": (textwrap.dedent(source).strip() + "\\n").splitlines(True)}\n        if kind == "code":\n            cell.update(execution_count=None, outputs=[])\n        cells.append(cell)\n\n    add("markdown", "intro", """\n    # Mission 3 ver29 — KLUE-ASL + Qwen3-Embedding / Kaggle\n\n    **처음에는 설정을 바꾸지 않고 screening을 실행하세요.**\n    Training 내부 fit 20,440개로 KLUE-ASL과 Qwen3-Embedding을 각각 최대 3 epoch 학습합니다.\n    development에서 기존 ver28, 두 새 단독 모델, 새 모델 1:1 평균, 기존·새 모델 4개 동일 평균을 비교합니다.\n    선택을 고정한 뒤 confirmation을 확인하며, threshold와 혼합 비율을 검색하지 않습니다.\n    전체 Training 학습은 자동 시작하지 않습니다. 결과 검토 후 별도 `full` 실행으로 진행합니다.\n\n    1. `mission3_ver29_inputs.zip`을 **Private Dataset**으로 업로드합니다.\n    2. 오른쪽 **Add Input**에서 데이터셋을 연결합니다. ZIP이 자동으로 풀려도 됩니다.\n       원본 JSON과 완료된 ver28의 작은 분석 ZIP이 함께 들어 있어, 처음에는 다른 Input이 필요 없습니다.\n    3. **Settings → Accelerator → GPU T4 ×2**, **Internet → On**을 선택합니다.\n    4. 아래 첫 코드 셀을 확인하고 **Save Version → Save & Run All**로 한 번 실행합니다.\n    5. 실행이 끝나면 저장된 버전의 **Output**에서 `ver29_analysis_*.zip`을 내려받습니다.\n\n    **주의:** 먼저 Run All로 학습한 뒤 Save & Run All을 누르면 처음부터 다시 학습합니다.\n    이 노트북은 GPU 한 개에서 새 모델 두 개를 순서대로 학습합니다. T4 두 개의 메모리를 합치지 않습니다.\n    T4에서는 FP16을 사용합니다. 이전 ver28의 BF16 에뮬레이션 선택을 수정했지만 실제 속도 향상은 실행 후 확인합니다.\n    설치·다운로드·평가·저장을 포함해 실행당 최대 8시간이며, 부족하면 checkpoint를 저장하고 `paused`로 종료합니다.\n    시간은 실제 배치 측정 후 안내합니다. Kaggle의 강제 종료나 가용량은 코드가 보장하지 않습니다.\n\n    점수 0.7 달성을 보장하지 않습니다. 기존 ver25~28·친구 모델·원본 데이터는 수정하지 않습니다.\n    외부 학습 데이터, 정답 변경, threshold/offset/앙상블 혼합비 검색은 하지 않습니다.\n    development와 confirmation에는 이전 실험의 사용 이력이 있습니다.\n    """)\n    add("code", "settings", """\n    # Edit only these fields. Keep defaults for the first run.\n    RUN_MODE = "screening"  # "screening" or "full"\n    REFERENCE_RUN = ""  # Empty: detect the bundled completed ver28 analysis ZIP\n    SOURCE_RUN = ""  # Full only: attached completed screening run folder\n    RESUME_RUN = ""  # Resume only: attached incomplete run folder of the same phase\n    INPUT_ZIP = ""  # Empty: detect the attached ZIP or extracted data directory\n    TIME_BUDGET_MINUTES = 480  # Maximum 480; installation time is included\n\n    import time\n    if type(TIME_BUDGET_MINUTES) is not int or not 30 <= TIME_BUDGET_MINUTES <= 480:\n        raise ValueError("TIME_BUDGET_MINUTES must be an integer between 30 and 480")\n    INVOCATION_STARTED = time.monotonic()\n    DEADLINE_MONOTONIC = INVOCATION_STARTED + TIME_BUDGET_MINUTES * 60\n    PREPARED = False\n    DEPENDENCIES_PREPARED = False\n    print("실행 모드:", RUN_MODE, "· 새 후보: KLUE-ASL + Qwen3-Embedding")\n    print("이번 실행의 최대 시간:", TIME_BUDGET_MINUTES, "분")\n    """)\n    add("code", "restore", """\n    # Restore the embedded package. No separate code upload is needed.\n    import importlib\n    import json\n    import os\n    from pathlib import Path\n    import sys\n    import tempfile\n    import uuid\n\n    RUNTIME_SOURCES = __SOURCES__\n    if "DEADLINE_MONOTONIC" not in globals():\n        raise RuntimeError("첫 설정 셀부터 실행하세요.")\n    if not Path("/kaggle/input").is_dir():\n        raise RuntimeError("이 노트북은 Kaggle용입니다. Kaggle에 업로드한 뒤 실행하세요.")\n    if time.monotonic() >= DEADLINE_MONOTONIC:\n        raise RuntimeError("실행 시간이 끝났습니다. 새 실행을 시작하려면 처음 셀부터 실행하세요.")\n    scratch_root = Path("/kaggle/temp")\n    scratch_root.mkdir(parents=True, exist_ok=True)\n    package_parent = Path(tempfile.mkdtemp(prefix="ver29_code_", dir=scratch_root))\n    package_dir = package_parent / "mission3_ver29"\n    package_dir.mkdir()\n    for name, source in RUNTIME_SOURCES.items():\n        (package_dir / name).write_text(source, encoding="utf-8")\n    # A fresh invocation uses exactly the embedded source revision.\n    for name in list(sys.modules):\n        if name == "mission3_ver29" or name.startswith("mission3_ver29."):\n            del sys.modules[name]\n    sys.path.insert(0, str(package_parent))\n    from mission3_ver29.settings import default_settings\n    from mission3_ver29.budget import run_supervised\n    SETTINGS = default_settings()\n    SETTINGS.update(run_mode=RUN_MODE, reference_run=REFERENCE_RUN, source_run=SOURCE_RUN,\n                    resume_run=RESUME_RUN, input_zip=INPUT_ZIP,\n                    output_dir="/kaggle/working/results/mission3/ver29",\n                    work_dir="/kaggle/temp/mission3_ver29", cache_dir="/kaggle/temp/hf_ver29",\n                    time_budget_minutes=TIME_BUDGET_MINUTES,\n                    run_validation=(RUN_MODE == "full"),\n                    invocation_started_monotonic=INVOCATION_STARTED,\n                    deadline_monotonic=DEADLINE_MONOTONIC)\n    environment = dict(os.environ)\n    environment["PYTHONPATH"] = str(package_parent) + os.pathsep + environment.get("PYTHONPATH", "")\n    environment["HF_HOME"] = SETTINGS["cache_dir"]\n    environment["TOKENIZERS_PARALLELISM"] = "false"\n    environment["HF_HUB_DISABLE_TELEMETRY"] = "1"\n    environment["PYTHONUNBUFFERED"] = "1"\n    PREPARED = True\n    print("코드 준비 완료 · 입력 ZIP 또는 자동 압축 해제된 데이터는 학습 전에 확인합니다.")\n    """.replace("__SOURCES__", repr(sources)))\n    add("code", "dependencies", """\n    # Keep Kaggle\'s compatible CUDA PyTorch. Install only unmet requirements.\n    from importlib.metadata import PackageNotFoundError, version\n    import re\n    if not globals().get("PREPARED"):\n        raise RuntimeError("코드 준비가 끝나지 않았습니다. 앞선 셀의 오류를 확인하세요.")\n    try:\n        torch_version = version("torch")\n    except PackageNotFoundError:\n        raise RuntimeError("Kaggle GPU 이미지의 PyTorch가 없습니다. GPU 설정을 확인하세요.") from None\n    match = re.match(r"^(\\\\d+)\\\\.(\\\\d+)", torch_version)\n    if not match or not (int(match[1]) == 2 and int(match[2]) >= 6):\n        raise RuntimeError("PyTorch 2.6 이상 3 미만의 Kaggle GPU 이미지가 필요합니다. "\n                           "CUDA 환경을 보호하기 위해 PyTorch를 자동 교체하지 않습니다.")\n    package_deadline = min(DEADLINE_MONOTONIC, time.monotonic() + 25 * 60)\n    run_supervised([sys.executable, "-m", "pip", "install", "--quiet",\n                    "--upgrade-strategy", "only-if-needed", "-r", str(package_dir / "requirements.txt")],\n                   deadline_monotonic=package_deadline, shutdown_margin_s=10,\n                   cwd=str(package_parent), env=environment)\n    DEPENDENCIES_PREPARED = True\n    print("패키지 준비 완료 · 남은 시간:", round((DEADLINE_MONOTONIC - time.monotonic()) / 60, 1), "분")\n    print({name: version(name) for name in ("torch", "transformers", "numpy", "scikit-learn")})\n    """)\n    add("code", "run", """\n    # Run the complete selected phase in a fresh Python process.\n    import zipfile\n    if not globals().get("PREPARED") or not globals().get("DEPENDENCIES_PREPARED"):\n        raise RuntimeError("준비가 완료되지 않았습니다. 처음 셀부터 실행하세요.")\n    settings_file = package_parent / "settings.json"\n    settings_file.write_text(json.dumps(SETTINGS, ensure_ascii=False, allow_nan=False), encoding="utf-8")\n    result_file = Path("/kaggle/working/ver29_last_run.json")\n    output_root = Path(SETTINGS["output_dir"])\n    output_root.mkdir(parents=True, exist_ok=True)\n    previous_run_states = set(output_root.glob("*/run_state.json"))\n    # Replace stale invocation pointers before launching; existing runs are retained.\n    result_file.write_text(json.dumps({"status": "starting", "run_dir": None}), encoding="utf-8")\n    failure = None\n    try:\n        run_supervised([sys.executable, "-u", "-m", "mission3_ver29.run_notebook",\n                        "--settings", str(settings_file), "--result-file", str(result_file)],\n                       deadline_monotonic=DEADLINE_MONOTONIC, shutdown_margin_s=10,\n                       cwd=str(package_parent), env=environment)\n    except Exception as error:\n        failure = type(error).__name__\n    try:\n        result = json.loads(result_file.read_text(encoding="utf-8"))\n    except (OSError, ValueError):\n        result = {"status": "interrupted", "submission_ready": False}\n        failure = failure or "ResultFileUnavailable"\n    if failure and not result.get("run_dir"):\n        new_run_states = set(output_root.glob("*/run_state.json")) - previous_run_states\n        if len(new_run_states) == 1:\n            # A hard kill does not prove that the resumable checkpoint is intact.\n            result.update(run_dir=str(next(iter(new_run_states)).parent),\n                          status="interrupted", submission_ready=False)\n            result_file.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")\n    print("실행 상태:", result.get("status", "확인 필요"))\n    print("이번 실행 시간:", round((time.monotonic() - INVOCATION_STARTED) / 60, 1), "분")\n    if result.get("run_dir"):\n        run_dir = Path(result["run_dir"])\n        print("결과 폴더:", run_dir)\n        selection_file = run_dir / "selection.json"\n        if selection_file.is_file():\n            selection = json.loads(selection_file.read_text(encoding="utf-8"))\n            for name, scores in selection.get("scores", {}).items():\n                print("Development", name, "Macro F1 =", round(scores["macro_f1"], 6))\n            print("고정한 선택:", selection.get("selected_recipe"))\n        validation_file = run_dir / "validation_metrics.json"\n        if validation_file.is_file():\n            validation = json.loads(validation_file.read_text(encoding="utf-8"))\n            print("공식 Validation Macro F1 =", round(validation["macro_f1"], 6))\n        archive = Path("/kaggle/working") / ("ver29_analysis_" + run_dir.name + ".zip")\n        allowed = {".json", ".csv", ".npz", ".npy", ".md", ".txt", ".py"}\n        excluded = {"checkpoints", "checkpoint", "resume_checkpoint", "best_model", "model", "tokenizer", "cache", "data"}\n        with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as handle:\n            for path in sorted(run_dir.rglob("*")):\n                relative = path.relative_to(run_dir)\n                if (path.is_file() and path.suffix.lower() in allowed\n                        and not any(part in excluded for part in relative.parts)\n                        and path.stat().st_size <= 20 * 1024 * 1024):\n                    handle.write(path, Path(run_dir.name) / relative)\n        print("분석용 ZIP:", archive)\n        print("분석 ZIP에는 큰 가중치와 이어 학습 checkpoint가 없습니다. 실행 폴더 전체도 보관하세요.")\n    if result.get("status") == "screening_complete":\n        print("후보 비교 완료. 분석 ZIP을 확인한 뒤 full 학습을 별도 실행하세요. 아직 제출 모델이 아닙니다.")\n    elif result.get("status") == "paused":\n        print("시간 제한으로 일시 중단했습니다. 저장된 Output을 Add Input으로 연결하고 RESUME_RUN에 실행 폴더를 넣으세요.")\n    elif result.get("status") == "complete" and result.get("submission_ready"):\n        print("전체 학습 및 제출 준비 완료. 최종 지표와 제출 manifest를 확인하세요.")\n    elif result.get("status") == "interrupted":\n        print("프로세스가 강제로 끝났습니다. checkpoint와 manifest 검증 전에는 재개 가능 여부를 확정할 수 없습니다.")\n    else:\n        print("완료 상태가 아닙니다. run_state.json과 오류 기록을 확인하세요.")\n    if result.get("error_type"):\n        print("오류 종류:", result["error_type"])\n    if failure:\n        raise RuntimeError("ver29 실행 오류: " + failure + ". 저장된 상태와 로그를 확인하세요.") from None\n    """)\n    add("markdown", "results", """\n    ## 결과와 다음 실행\n\n    출력 위치는 `/kaggle/working/results/mission3/ver29/실행시각_고유번호/`입니다.\n    저장된 버전의 Output에서 `ver29_analysis_*.zip`을 내려받아 분석에 사용합니다.\n    분석 ZIP은 작은 지표·예측·코드 모음이며 가중치의 백업이나 재개용 파일이 아닙니다.\n\n    | 상태 | 의미 | 다음 행동 |\n    |---|---|---|\n    | `screening_complete` | 후보 비교와 선택 완료 | 결과 검토 후 별도 full 실행 |\n    | `paused` | 예산에 맞춰 중간 저장 | 이전 Output 연결 후 같은 모드로 재개 |\n    | `complete` + `submission_ready=true` | 전체 학습·제출 준비 완료 | 공식 Validation 지표와 제출 파일 확인 |\n    | `failed` | 오류 | 오류 기록 확인; 완료 모델로 사용하지 않기 |\n    | `interrupted` | 강제 종료로 저장 완전성 미확인 | 로그·checkpoint 검증 후 재개 여부 판단 |\n\n    **이어 실행:** 새 노트북에 이전 저장 버전의 Output을 Add Input으로 연결합니다.\n    `RESUME_RUN`에 그 안의 `run_state.json`이 있는 실행 폴더를 지정합니다.\n    `RUN_MODE`는 이전과 같게 유지합니다. ver29 입력 데이터셋도 연결해야 합니다.\n    입력 폴더는 읽기 전용이며, 재개 결과는 새 실행 폴더에 저장합니다.\n\n    **전체 학습:** 결과를 검토한 뒤 완료된 screening Output을 새 노트북에 연결합니다.\n    `RUN_MODE="full"`, `SOURCE_RUN="/kaggle/input/.../results/mission3/ver29/실행폴더"`,\n    `RESUME_RUN=""`로 바꾸고 Save Version → Save & Run All을 실행합니다.\n    development에서 고정한 모델 구성과 epoch로 선택된 1개·2개·4개 모델을 전체 Training으로 학습하며,\n    공식 Validation 정답은 학습에 사용하지 않습니다. 실행당 8시간 제한과 재개 기능은 동일합니다.\n\n    Save & Run All은 새 실행입니다. 이미 실행 중인 버전에 반복해서 누르지 마세요.\n    대화형 세션을 켰다면 별도로 종료하고, 버전 저장 완료와 Output 파일을 확인하세요.\n    """)\n    return {"cells": cells,\n            "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},\n                         "language_info": {"name": "python", "version": "3.11"}, "accelerator": "GPU"},\n            "nbformat": 4, "nbformat_minor": 5}\n\n\ndef build(output=None):\n    target = Path(output) if output else HERE.parent / "mission3_ver29.ipynb"\n    notebook = make_notebook()\n    for cell in notebook["cells"]:\n        if cell["cell_type"] == "code":\n            compile("".join(cell["source"]), cell["id"], "exec")\n    target.write_text(json.dumps(notebook, ensure_ascii=False, indent=1) + "\\n", encoding="utf-8")\n    return target\n\n\nif __name__ == "__main__":\n    print(build())\n', 'data.py': '"""Text-only data and label-balanced, indivisible Training transcript groups."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nimport re\nimport tempfile\nimport unicodedata\n\nimport numpy as np\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nTARGET_SYMPTOMS = (\n    "고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란",\n)\nLEXICAL_PATTERNS = {\n    "nausea": r"오심|메스꺼|메스껍|메슥|매스꺼|매스껍|매슥|울렁|구역|미식|느글|토할",\n    "question": r"\\?|나요|까요|습니까|신가|세요",\n    "negation": r"아니|없|않|못|안\\s|아닌",\n}\nLEXICAL_COUNTS_NOTE = (\n    "Regex matches are review candidates only: questions and negation may appear elsewhere "\n    "in the dialogue. These flags do not establish annotation errors or modify labels/predictions."\n)\n\n\ndef dialogue_text(annotation: dict) -> str:\n    """Read only utterance text, keeping turn boundaries and chronological order."""\n    key = "utterances" if "utterances" in annotation else "utterences"\n    utterances = annotation.get(key)\n    if not isinstance(utterances, list):\n        raise ValueError("utterances/utterences must be a list")\n    texts = []\n    for utterance in utterances:\n        if not isinstance(utterance, dict) or not isinstance(utterance.get("text"), str):\n            raise ValueError("Each utterance must be an object with string text")\n        texts.append(utterance["text"])\n    return "\\n".join(texts)\n\n\ndef target_vector(annotation: dict) -> list[int]:\n    if "symptom" not in annotation:\n        raise ValueError("Training/evaluation JSON has no symptom target")\n    raw = annotation["symptom"]\n    if raw is None:\n        raw = []\n    if isinstance(raw, str):\n        raw = [raw]\n    if not isinstance(raw, list) or any(not isinstance(item, str) for item in raw):\n        raise ValueError("symptom must be a string, a list of strings, or null")\n    found = {item.strip() for item in raw}\n    return [int(name in found) for name in TARGET_SYMPTOMS]\n\n\ndef text_group(text: str) -> str:\n    normalized = " ".join(unicodedata.normalize("NFC", text).split())\n    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()\n\n\ndef load_dataset(label_dir, include_targets: bool = False):\n    label_dir = Path(label_dir)\n    if not label_dir.is_dir():\n        raise ValueError(f"Label directory does not exist: {label_dir}")\n    paths = sorted(path for path in label_dir.rglob("*.json") if path.is_file())\n    if not paths:\n        raise ValueError(f"No label JSON files: {label_dir}")\n    if len({path.name.casefold() for path in paths}) != len(paths):\n        raise ValueError("Duplicate JSON filenames make submission identifiers ambiguous")\n\n    def read(path):\n        with path.open(encoding="utf-8-sig") as handle:\n            annotation = json.load(handle)\n        if not isinstance(annotation, dict):\n            raise ValueError(f"Expected a JSON object: {path}")\n        # Inference never inspects symptom, even when a supplied value is malformed.\n        return dialogue_text(annotation), target_vector(annotation) if include_targets else None\n\n    with ThreadPoolExecutor(max_workers=8) as executor:\n        rows = list(executor.map(read, paths))\n    texts = [row[0] for row in rows]\n    targets = np.asarray([row[1] for row in rows], dtype=np.int8) if include_targets else None\n    return paths, texts, targets\n\n\ndef _binary(values, rows=None, name="predictions"):\n    values = np.asarray(values)\n    if (values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS)\n            or (rows is not None and values.shape[0] != rows)\n            or not np.isin(values, [0, 1]).all()):\n        raise ValueError(f"{name} must have shape (samples, 9) and contain only discrete 0/1 values")\n    return values.astype(np.int8)\n\n\ndef _assign_groups(counts, sizes, proportions, seed):\n    """Iteratively allocate the rarest remaining group label, weighted by rows.\n\n    Positive and negative counts are balanced together. A transcript group is\n    indivisible, including when its original rows have conflicting labels.\n    """\n    rng = np.random.default_rng(seed)\n    presence = counts > 0\n    remaining = presence.sum(axis=0)\n    label_deficits = proportions[:, None] * counts.sum(axis=0)\n    row_deficits = proportions * sizes.sum()\n    active = np.ones(len(sizes), dtype=bool)\n    assignment = np.full(len(sizes), -1, dtype=np.int8)\n    coverage = np.zeros((len(proportions), counts.shape[1]), dtype=bool)\n    assigned_counts = np.zeros(len(proportions), dtype=int)\n    unassigned_count = len(sizes)\n    while unassigned_count:\n        available = np.flatnonzero(remaining > 0)\n        rare = available[remaining[available] == remaining[available].min()]\n        label = int(rng.choice(rare))\n        candidates = np.flatnonzero(active & presence[:, label])\n        # Large indivisible groups are placed while there is still capacity.\n        rng.shuffle(candidates)\n        candidates = candidates[np.argsort(-sizes[candidates], kind="stable")]\n        for group in candidates:\n            sides = np.flatnonzero(row_deficits > 0)\n            if not len(sides):\n                sides = np.arange(len(proportions))\n            missing = np.flatnonzero(~coverage[:, label])\n            if len(missing) and remaining[label] <= len(missing):\n                sides = missing\n            # Every split must contain a group, even for very small datasets.\n            empty = np.flatnonzero(assigned_counts == 0)\n            if unassigned_count == len(empty):\n                sides = empty\n            sides = sides[label_deficits[sides, label] == label_deficits[sides, label].max()]\n            sides = sides[row_deficits[sides] == row_deficits[sides].max()]\n            side = int(rng.choice(sides))\n            assignment[group] = side\n            row_deficits[side] -= sizes[group]\n            label_deficits[side] -= counts[group]\n            remaining -= presence[group]\n            coverage[side] |= presence[group]\n            active[group] = False\n            assigned_counts[side] += 1\n            unassigned_count -= 1\n    return assignment\n\n\ndef dataset_digest(texts, targets):\n    """Hash ordered exact model inputs and original nine-column targets."""\n    targets = _binary(targets, len(texts), "targets")\n    digest = hashlib.sha256()\n    digest.update(json.dumps(TARGET_SYMPTOMS, ensure_ascii=False).encode("utf-8"))\n    for text, row in zip(texts, targets):\n        encoded = text.encode("utf-8")\n        digest.update(len(encoded).to_bytes(8, "little"))\n        digest.update(encoded)\n        digest.update(row.tobytes())\n    return digest.hexdigest()\n\n\ndef three_way_split(texts, targets, seed: int = 42):\n    """Return fit/development/confirmation indices and an aggregate manifest.\n\n    Eight deterministic allocation attempts minimize only split label/row\n    imbalance. Model predictions and official Validation never participate.\n    No row or original label is dropped, masked, or changed.\n    """\n    targets = _binary(targets, len(texts), "targets")\n    if type(seed) is not int or seed < 0:\n        raise ValueError("seed must be a nonnegative integer")\n    if any(not isinstance(text, str) for text in texts):\n        raise TypeError("texts must contain strings only")\n    groups = np.asarray([text_group(text) for text in texts])\n    unique, inverse, sizes = np.unique(groups, return_inverse=True, return_counts=True)\n    if len(unique) < 3:\n        raise ValueError("Three-way split requires at least three distinct normalized text groups")\n    positive = np.zeros((len(unique), len(TARGET_SYMPTOMS)), dtype=np.int64)\n    np.add.at(positive, inverse, targets)\n    counts = np.concatenate([positive, sizes[:, None] - positive], axis=1)\n    proportions = np.asarray([.70, .15, .15])\n    totals = counts.sum(axis=0)\n    best = None\n    for attempt in range(8):\n        assignment = _assign_groups(counts, sizes, proportions, seed + attempt)\n        splits = tuple(np.flatnonzero(assignment[inverse] == side) for side in range(3))\n        observed = np.asarray([counts[assignment == side].sum(axis=0) for side in range(3)])\n        missing = int(((observed == 0) & (totals > 0)).sum())\n        imbalance = float(np.abs(observed / np.maximum(totals, 1) - proportions[:, None]).mean()\n                          + np.abs(np.asarray([len(ids) for ids in splits]) / len(texts) - proportions).mean())\n        rank = (missing, imbalance)\n        if best is None or rank < best[0]:\n            best = (rank, splits, assignment, attempt)\n    _, splits, assignment, attempt = best\n    data_hash = dataset_digest(texts, targets)\n    split_hash = hashlib.sha256(data_hash.encode("ascii") + assignment[inverse].tobytes()).hexdigest()\n    report = {\n        "method": "iterative multilabel stratification of indivisible normalized text groups",\n        "group_normalization": "NFC and whitespace collapse; model input text is preserved",\n        "seed": seed, "allocation_attempt": attempt, "label_order": list(TARGET_SYMPTOMS),\n        "data_digest": data_hash, "split_digest": split_hash,\n        "samples": len(texts), "text_groups": len(unique),\n        "duplicate_groups": int((sizes > 1).sum()),\n        "duplicate_extra_samples": int((sizes - 1).sum()),\n        "conflicting_label_groups": int(np.any((positive > 0) & (positive < sizes[:, None]), axis=1).sum()),\n        "all_rows_preserved": True, "cross_split_text_groups": 0,\n        "positive_counts": dict(zip(TARGET_SYMPTOMS, targets.sum(axis=0).astype(int).tolist())),\n        "splits": {}, "coverage_warnings": [],\n        "limitations": "Group constraints may prevent exact fractions or label coverage. Near duplicates are not checked. "\n                       "Training has prior experiment exposure; this is not a new external test set.",\n    }\n    for side, (name, ids) in enumerate(zip(("fit", "development", "confirmation"), splits)):\n        positives = targets[ids].sum(axis=0)\n        report["splits"][name] = {\n            "samples": len(ids), "text_groups": int((assignment == side).sum()),\n            "requested_fraction": float(proportions[side]), "actual_fraction": len(ids) / len(texts),\n            "indices_digest": hashlib.sha256(ids.astype("<i8").tobytes()).hexdigest(),\n            "group_digest": hashlib.sha256("".join(unique[assignment == side]).encode("ascii")).hexdigest(),\n            "positive_counts": dict(zip(TARGET_SYMPTOMS, positives.astype(int).tolist())),\n            "negative_counts": dict(zip(TARGET_SYMPTOMS, (len(ids) - positives).astype(int).tolist())),\n            "positive_prevalence": dict(zip(TARGET_SYMPTOMS, (positives / len(ids)).tolist())),\n        }\n        for index, symptom in enumerate(TARGET_SYMPTOMS):\n            if positives[index] in (0, len(ids)):\n                report["coverage_warnings"].append({"split": name, "symptom": symptom,\n                    "missing_class": "positive" if positives[index] == 0 else "negative"})\n    return *splits, report\n\n\ndef _subset_groups_at_capacity(assignment, sizes, capacity, seed):\n    """Keep the preferred groups first, while reaching the largest feasible size."""\n    rng = np.random.default_rng(seed)\n    order = rng.permutation(len(sizes))\n    order = order[np.argsort(assignment[order], kind="stable")]\n    # Bit i indicates whether i rows can be formed from whole remaining groups.\n    # Suffix reachability lets us preserve preferred groups without underfilling\n    # a feasible request merely because one large group was selected first.\n    mask = (1 << (capacity + 1)) - 1\n    reachable = [0] * (len(order) + 1)\n    reachable[-1] = 1\n    for position in range(len(order) - 1, -1, -1):\n        tail = reachable[position + 1]\n        reachable[position] = (tail | (tail << int(sizes[order[position]]))) & mask\n    remaining = reachable[0].bit_length() - 1\n    selected = np.zeros(len(sizes), dtype=bool)\n    for position, group in enumerate(order):\n        after = remaining - int(sizes[group])\n        if after >= 0 and (reachable[position + 1] >> after) & 1:\n            selected[group] = True\n            remaining = after\n    return selected\n\n\ndef sample_fit_subset(texts, targets, fit, sample_count: int = 4000, seed: int = 42):\n    """Select a reproducible, label-balanced pilot from fit groups only.\n\n    Original texts, targets, and the existing three-way split are unchanged.\n    If indivisible duplicates prevent the exact request, return the largest\n    feasible count below it and record the shortfall. No model predictions or\n    development/confirmation labels influence the allocation.\n    """\n    targets = _binary(targets, len(texts), "targets")\n    if type(sample_count) is not int or sample_count <= 0:\n        raise ValueError("sample_count must be a positive integer")\n    if type(seed) is not int or seed < 0:\n        raise ValueError("seed must be a nonnegative integer")\n    if any(not isinstance(text, str) for text in texts):\n        raise TypeError("texts must contain strings only")\n    fit = np.asarray(fit)\n    if (fit.ndim != 1 or not np.issubdtype(fit.dtype, np.integer)\n            or len(np.unique(fit)) != len(fit)\n            or np.any(fit < 0) or np.any(fit >= len(texts))):\n        raise ValueError("fit must contain unique valid integer indices")\n    if len(fit) < sample_count:\n        raise ValueError("sample_count exceeds the available fit samples")\n    fit = np.sort(fit.astype(np.int64))\n    groups = np.asarray([text_group(text) for text in texts])\n    all_groups, all_inverse, all_sizes = np.unique(groups, return_inverse=True, return_counts=True)\n    fit_sizes = np.bincount(all_inverse[fit], minlength=len(all_groups))\n    if np.any((fit_sizes > 0) & (fit_sizes != all_sizes)):\n        raise ValueError("fit must contain complete normalized text groups to prevent split leakage")\n    unique, inverse, sizes = np.unique(groups[fit], return_inverse=True, return_counts=True)\n    positive = np.zeros((len(unique), len(TARGET_SYMPTOMS)), dtype=np.int64)\n    np.add.at(positive, inverse, targets[fit])\n    counts = np.concatenate([positive, sizes[:, None] - positive], axis=1)\n    parent_positive = targets[fit].sum(axis=0)\n    parent_prevalence = parent_positive / len(fit)\n    parent_counts = counts.sum(axis=0)\n    proportions = np.asarray([sample_count / len(fit), 1 - sample_count / len(fit)])\n    best = None\n    for attempt in range(8):\n        if sample_count == len(fit):\n            preferred = np.zeros(len(sizes), dtype=np.int8)\n        else:\n            preferred = _assign_groups(counts, sizes, proportions, seed + attempt)\n        selected = _subset_groups_at_capacity(preferred, sizes, sample_count, seed + attempt)\n        indices = fit[selected[inverse]]\n        if not len(indices):\n            raise ValueError("No complete fit text group fits within sample_count")\n        observed = counts[selected].sum(axis=0)\n        missing = int(((observed == 0) & (parent_counts > 0)).sum())\n        deviation = float(np.abs(targets[indices].mean(axis=0) - parent_prevalence).mean())\n        rank = (missing, deviation)\n        if best is None or rank < best[0]:\n            best = (rank, indices, selected, attempt)\n    _, indices, selected, attempt = best\n    data_hash = dataset_digest(texts, targets)\n    parent_hash = hashlib.sha256(fit.astype("<i8").tobytes()).hexdigest()\n    indices_hash = hashlib.sha256(indices.astype("<i8").tobytes()).hexdigest()\n    subset_hash = hashlib.sha256((data_hash + parent_hash + indices_hash).encode("ascii")).hexdigest()\n    observed_positive = targets[indices].sum(axis=0)\n    observed_prevalence = observed_positive / len(indices)\n    report = {\n        "method": "iterative multilabel stratification within fit, with exact whole-group capacity",\n        "group_normalization": "NFC and whitespace collapse; model input text is preserved",\n        "seed": seed, "allocation_attempt": attempt, "label_order": list(TARGET_SYMPTOMS),\n        "requested_samples": sample_count, "actual_samples": len(indices),\n        "parent_fit_samples": len(fit), "unfilled_samples": sample_count - len(indices),\n        "exact_sample_count": len(indices) == sample_count,\n        "data_digest": data_hash, "parent_fit_digest": parent_hash,\n        "indices_digest": indices_hash, "subset_digest": subset_hash,\n        "parent_fit_text_groups": len(unique), "selected_text_groups": int(selected.sum()),\n        "selected_duplicate_groups": int(((sizes > 1) & selected).sum()),\n        "selected_conflicting_label_groups": int((np.any(\n            (positive > 0) & (positive < sizes[:, None]), axis=1) & selected).sum()),\n        "outside_fit_samples": 0, "original_targets_preserved": True,\n        "positive_counts": dict(zip(TARGET_SYMPTOMS, observed_positive.astype(int).tolist())),\n        "negative_counts": dict(zip(TARGET_SYMPTOMS, (len(indices) - observed_positive).astype(int).tolist())),\n        "positive_prevalence": dict(zip(TARGET_SYMPTOMS, observed_prevalence.tolist())),\n        "parent_positive_counts": dict(zip(TARGET_SYMPTOMS, parent_positive.astype(int).tolist())),\n        "parent_positive_prevalence": dict(zip(TARGET_SYMPTOMS, parent_prevalence.tolist())),\n        "prevalence_deviation": dict(zip(TARGET_SYMPTOMS,\n                                         (observed_prevalence - parent_prevalence).tolist())),\n        "coverage": {}, "coverage_warnings": [],\n        "limitations": "Indivisible groups may prevent exact sample counts or class coverage. "\n                       "This pilot subset does not establish full-data performance.",\n    }\n    for index, symptom in enumerate(TARGET_SYMPTOMS):\n        positive_present = bool(observed_positive[index] > 0)\n        negative_present = bool(observed_positive[index] < len(indices))\n        report["coverage"][symptom] = {"positive": positive_present, "negative": negative_present}\n        if not positive_present and parent_positive[index] > 0:\n            report["coverage_warnings"].append({"symptom": symptom, "missing_class": "positive"})\n        if not negative_present and parent_positive[index] < len(fit):\n            report["coverage_warnings"].append({"symptom": symptom, "missing_class": "negative"})\n    return indices, report\n\n\ndef _atomic_text(path, write, encoding="utf-8-sig"):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = None\n    try:\n        with tempfile.NamedTemporaryFile(mode="w", encoding=encoding, newline="",\n                                         dir=path.parent, prefix=".write_", suffix=".tmp", delete=False) as handle:\n            temporary = Path(handle.name)\n            write(handle)\n        temporary.replace(path)\n    finally:\n        if temporary is not None:\n            temporary.unlink(missing_ok=True)\n\n\ndef write_json(path, value):\n    _atomic_text(path, lambda handle: json.dump(value, handle, ensure_ascii=False, indent=2, allow_nan=False),\n                 encoding="utf-8")\n\n\ndef write_predictions(paths, predictions, output):\n    """Write already discrete model decisions; this API has no score or threshold."""\n    predictions = _binary(predictions, len(paths))\n    if len({Path(path).name.casefold() for path in paths}) != len(paths):\n        raise ValueError("Duplicate prediction identifiers")\n\n    def write(handle):\n        writer = csv.writer(handle)\n        writer.writerow(["label file name", "symptom"])\n        for path, flags in zip(paths, predictions):\n            writer.writerow([Path(path).name, str([name for name, flag in zip(TARGET_SYMPTOMS, flags) if flag])])\n    _atomic_text(output, write)\n    return predictions\n\n\ndef score_predictions(targets, predictions):\n    targets = _binary(targets, name="targets")\n    predictions = _binary(predictions, len(targets))\n    if not len(targets):\n        raise ValueError("Cannot score an empty dataset")\n    precision, recall, f1, support = precision_recall_fscore_support(targets, predictions, zero_division=0)\n    return {\n        "macro_f1": float(f1_score(targets, predictions, average="macro", zero_division=0)),\n        "micro_f1": float(f1_score(targets, predictions, average="micro", zero_division=0)),\n        "samples": int(len(targets)),\n        "per_symptom": {name: {"f1": float(f1[i]), "precision": float(precision[i]),\n                                "recall": float(recall[i]), "support": int(support[i]),\n                                "false_positive": int(((targets[:, i] == 0) & (predictions[:, i] == 1)).sum()),\n                                "false_negative": int(((targets[:, i] == 1) & (predictions[:, i] == 0)).sum())}\n                        for i, name in enumerate(TARGET_SYMPTOMS)},\n    }\n\n\ndef training_diagnostics(texts, targets):\n    """Return aggregate counts only, including suspected nausea annotation ambiguity."""\n    targets = _binary(targets, len(texts), "targets")\n    groups = {}\n    for index, text in enumerate(texts):\n        groups.setdefault(text_group(text), []).append(index)\n    duplicates = [indices for indices in groups.values() if len(indices) > 1]\n    conflicts = [indices for indices in duplicates if len(np.unique(targets[indices], axis=0)) > 1]\n    nausea = targets[:, TARGET_SYMPTOMS.index("오심")].astype(bool)\n    vomiting = targets[:, TARGET_SYMPTOMS.index("구토")].astype(bool)\n    signals = {key: np.asarray([bool(re.search(pattern, text)) for text in texts])\n               for key, pattern in LEXICAL_PATTERNS.items()}\n    lexical = {key: int(flags.sum()) for key, flags in signals.items()}\n    for key in ("question", "negation"):\n        cooccurrence = signals["nausea"] & signals[key]\n        lexical[f"nausea_and_{key}"] = int(cooccurrence.sum())\n        lexical[f"nausea_and_{key}_label_positive"] = int((cooccurrence & nausea).sum())\n        lexical[f"nausea_and_{key}_label_negative"] = int((cooccurrence & ~nausea).sum())\n    lexical["nausea_mention_label_positive"] = int((signals["nausea"] & nausea).sum())\n    lexical["nausea_mention_label_negative"] = int((signals["nausea"] & ~nausea).sum())\n    return {\n        "samples": len(texts), "all_zero_samples": int((targets.sum(axis=1) == 0).sum()),\n        "empty_text_samples": sum(not text.strip() for text in texts),\n        "positive_counts": {name: int(targets[:, i].sum()) for i, name in enumerate(TARGET_SYMPTOMS)},\n        "text_groups": len(groups), "duplicate_groups": len(duplicates),\n        "duplicate_extra_samples": sum(len(indices) - 1 for indices in duplicates),\n        "conflicting_label_groups": len(conflicts),\n        "conflicting_label_samples": sum(len(indices) for indices in conflicts),\n        "nausea_vomiting_contingency": {\n            "neither": int((~nausea & ~vomiting).sum()), "nausea_only": int((nausea & ~vomiting).sum()),\n            "vomiting_only": int((~nausea & vomiting).sum()), "both": int((nausea & vomiting).sum())},\n        "lexical_cooccurrence_counts": lexical,\n        "lexical_patterns": LEXICAL_PATTERNS,\n        "lexical_counts_note": LEXICAL_COUNTS_NOTE,\n    }\n\n\n', 'inference.py': '"""Offline, text-only inference from a completed ver29 run directory."""\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\nimport sys\n\n# Also support the competition\'s direct `python inference.py` entrypoint.\nif __package__ in (None, ""):\n    sys.path.insert(0, str(Path(__file__).resolve().parent.parent))\n    __package__ = Path(__file__).resolve().parent.name\n\nimport numpy as np\n\nfrom . import analysis, data, trainer\nfrom .runtime import FORMAT, RECIPES, find_run, read_json\n\n\ndef predict(label_dir, run_dir, output, device="cpu", batch_size=8):\n    checkpoint = Path(run_dir).expanduser().resolve()\n    if checkpoint.is_file():\n        if checkpoint.name != \'submission_manifest.json\':\n            raise ValueError("ckpt_path must be the complete run folder or its submission_manifest.json")\n        run_dir = checkpoint.parent\n    root = find_run(run_dir)\n    if read_json(root / "run_state.json")["status"] != "complete":\n        raise ValueError("Inference requires a completed full-training run")\n    manifest = read_json(root / "submission_manifest.json")\n    if (manifest.get("format") != FORMAT or manifest.get("label_order") != list(data.TARGET_SYMPTOMS)\n            or manifest.get("decision_threshold") != .5):\n        raise ValueError("Submission format, label order, or decision policy mismatch")\n    members = manifest.get("members", {})\n    if not isinstance(members, dict) or list(members) not in RECIPES.values():\n        raise ValueError("Invalid submission ensemble")\n    paths, texts, _ = data.load_dataset(label_dir, include_targets=False)\n    predictions = []\n    for relative in members.values():\n        checkpoint = (root / relative).resolve()\n        if not checkpoint.is_relative_to(root):\n            raise ValueError("Model path escapes the submission folder")\n        predictions.append(trainer.predict_checkpoint(checkpoint, texts, device=device, batch_size=batch_size))\n    logits = analysis.equal_mean_logits(predictions)\n    data.write_predictions(paths, (logits >= 0).astype(np.int8), output)\n    return Path(output)\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--label-dir", "--label_dir", dest="label_dir", required=True)\n    parser.add_argument("--run-dir", "--ckpt_path", dest="run_dir", required=True)\n    parser.add_argument("--audio_dir", help="Accepted for the competition interface; audio is not read")\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--device", default=None)\n    parser.add_argument("--batch-size", type=int, default=8)\n    args = parser.parse_args(argv)\n    import torch\n    device = args.device or ("cuda" if torch.cuda.is_available() else "cpu")\n    print(predict(args.label_dir, args.run_dir, args.output, device, args.batch_size))\n\n\nif __name__ == "__main__":\n    main()\n', 'package_inputs.py': '"""Build verified labels and a completed ver28 analysis for a private Dataset.\n\nOriginal JSON bytes are copied unchanged. Model checkpoints, cached predictions,\nand unrelated data are excluded except the explicitly supplied analysis ZIP.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport tempfile\nfrom typing import Any\nimport zipfile\n\nfrom .bootstrap import TRAIN_PREFIX, VALID_PREFIX, _relative, _selected_files\n\n\nMANIFEST_NAME = "bundle_manifest.json"\nDEFAULT_COUNTS = (29_200, 3_640)\nREFERENCE_PREFIX = "ver28_reference/"\n\n\ndef _sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as reader:\n        for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _counts(files: list[dict[str, Any]]) -> tuple[int, int]:\n    return tuple(sum(row["path"].startswith(prefix) for row in files)\n                 for prefix in (TRAIN_PREFIX, VALID_PREFIX))\n\n\ndef _check_counts(files: list[dict[str, Any]], expected: tuple[int, int]) -> None:\n    if _counts(files) != expected:\n        raise ValueError(f"Label counts differ: expected {expected}, found {_counts(files)}")\n\n\ndef _zip_info(name: str) -> zipfile.ZipInfo:\n    info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))\n    info.create_system = 3\n    info.external_attr = 0o100644 << 16\n    info.compress_type = zipfile.ZIP_DEFLATED\n    return info\n\n\ndef _validate_reference(raw: bytes) -> None:\n    """Check that the unchanged ZIP identifies one completed screening run."""\n    with zipfile.ZipFile(io.BytesIO(raw)) as archive:\n        names = archive.namelist()\n        for info in archive.infolist():\n            _relative(info.orig_filename)\n        states = [name for name in names if name.endswith("/run_state.json")]\n        if len(states) != 1:\n            raise ValueError("Reference analysis must identify one completed ver28 screening run")\n        state = json.loads(archive.read(states[0]))\n        if (state.get("format") != "mission3_ver28_v1"\n                or state.get("run_mode") != "screening" or state.get("status") != "screening_complete"):\n            raise ValueError("Reference analysis must be a completed ver28 screening run")\n        prefix = states[0].rsplit("/", 1)[0] + "/"\n        required = {prefix + name for name in ("split_indices.npz", "split_manifest.json",\n                                               "development_predictions.npz", "confirmation_predictions.npz",\n                                               "selection.json", "artifact_manifest.json")}\n        if not required.issubset(names):\n            raise ValueError("Reference analysis is missing screening provenance or predictions")\n\n\ndef verify_package(path: Path, expected_counts: tuple[int, int] = DEFAULT_COUNTS) -> dict[str, Any]:\n    """Re-read every label and require its content to match the stored manifest."""\n    with zipfile.ZipFile(path) as archive:\n        files, signature = _selected_files(archive)\n        _check_counts(files, expected_counts)\n        manifest = json.loads(archive.read(MANIFEST_NAME))\n        reference = manifest.get("reference_analysis")\n        allowed = {row["path"] for row in files} | {MANIFEST_NAME}\n        if reference is not None:\n            if not isinstance(reference, dict) or set(reference) != {"path", "size", "sha256"}:\n                raise ValueError("Invalid reference analysis metadata")\n            name = reference["path"]\n            _relative(name)\n            if (not name.startswith(REFERENCE_PREFIX) or not name.endswith(".zip")\n                    or len(name.split("/")) != 2):\n                raise ValueError("Reference analysis must be one ZIP under ver28_reference")\n            raw = archive.read(name)\n            if len(raw) != reference["size"] or hashlib.sha256(raw).hexdigest() != reference["sha256"]:\n                raise ValueError("Reference analysis content differs from manifest")\n            _validate_reference(raw)\n            allowed.add(name)\n        if set(archive.namelist()) != allowed:\n            raise ValueError("Compact input archive must contain only labels, its manifest, and the declared reference")\n        if len(manifest["files"]) != len(files):\n            raise ValueError("Compact manifest must describe labels only")\n        for row in files:\n            raw = archive.read(row["path"])\n            if len(raw) != row["size"] or hashlib.sha256(raw).hexdigest() != row["sha256"]:\n                raise ValueError("Label content differs from manifest")\n        return {\n            "training_count": expected_counts[0],\n            "validation_count": expected_counts[1],\n            "label_file_count": len(files),\n            "label_uncompressed_bytes": sum(row["size"] for row in files),\n            "manifest_sha256": signature,\n            "reference_analysis": reference,\n            "verification": "all_label_and_declared_reference_sizes_and_sha256_match",\n        }\n\n\ndef package_inputs(source: Path, destination: Path,\n                   expected_counts: tuple[int, int] = DEFAULT_COUNTS,\n                   *, reference_analysis: Path | None = None) -> dict[str, Any]:\n    """Create a new archive atomically; refuse to replace any existing output."""\n    source = Path(source).expanduser().resolve()\n    if not source.is_file():\n        raise FileNotFoundError(f"Source archive is missing: {source}")\n    requested_destination = Path(destination).expanduser()\n    if requested_destination.is_symlink():\n        raise ValueError("Output archive cannot be a symlink")\n    destination = requested_destination.resolve()\n    if source == destination or destination.exists():\n        raise FileExistsError("Output already exists; choose a new output path")\n    if any(type(count) is not int or count < 1 for count in expected_counts) or len(expected_counts) != 2:\n        raise ValueError("Expected Training and Validation counts must be positive integers")\n    before = source.stat()\n    reference = Path(reference_analysis).expanduser().resolve() if reference_analysis is not None else None\n    reference_bytes = reference.read_bytes() if reference is not None else None\n    reference_metadata = None\n    if reference is not None:\n        if reference == destination or reference == source:\n            raise ValueError("Reference analysis must be separate from the input and output archives")\n        _validate_reference(reference_bytes)\n        reference_metadata = {"path": REFERENCE_PREFIX + reference.name, "size": len(reference_bytes),\n                              "sha256": hashlib.sha256(reference_bytes).hexdigest()}\n    temporary: Path | None = None\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    try:\n        with zipfile.ZipFile(source) as original:\n            files, source_signature = _selected_files(original)\n            _check_counts(files, expected_counts)\n            manifest = {\n                "format": "mission3_ver5_inputs_v1",\n                "producer": "mission3_ver29.package_inputs",\n                "privacy": "private_dataset_only",\n                "content": "original_label_json_bytes_and_declared_ver28_analysis",\n                "training_count": expected_counts[0],\n                "validation_count": expected_counts[1],\n                "files": files,\n            }\n            if reference_metadata is not None:\n                manifest["reference_analysis"] = reference_metadata\n            with tempfile.NamedTemporaryFile(dir=destination.parent, prefix=".ver29_inputs_",\n                                             suffix=".tmp", delete=False) as writer:\n                temporary = Path(writer.name)\n            with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED,\n                                 compresslevel=6, allowZip64=True) as compact:\n                for row in files:\n                    raw = original.read(row["path"])\n                    if len(raw) != row["size"] or hashlib.sha256(raw).hexdigest() != row["sha256"]:\n                        raise ValueError("Source label content differs from manifest")\n                    compact.writestr(_zip_info(row["path"]), raw, compresslevel=6)\n                if reference_metadata is not None:\n                    compact.writestr(_zip_info(reference_metadata["path"]), reference_bytes, compresslevel=6)\n                raw_manifest = json.dumps(manifest, ensure_ascii=False, sort_keys=True,\n                                          separators=(",", ":")).encode("utf-8")\n                compact.writestr(_zip_info(MANIFEST_NAME), raw_manifest, compresslevel=6)\n            source_label_compressed_bytes = sum(original.getinfo(row["path"]).compress_size for row in files)\n        verification = verify_package(temporary, expected_counts)\n        after = source.stat()\n        if (before.st_size, before.st_mtime_ns) != (after.st_size, after.st_mtime_ns):\n            raise ValueError("Source archive changed while packaging")\n        compact_bytes = temporary.stat().st_size\n        report = {\n            "format": "mission3_ver29_input_packaging_report_v1",\n            "source_path": str(source),\n            "source_archive_bytes": before.st_size,\n            "source_manifest_sha256": source_signature,\n            "source_label_compressed_bytes": source_label_compressed_bytes,\n            "output_path": str(destination),\n            "output_archive_bytes": compact_bytes,\n            "output_archive_sha256": _sha256(temporary),\n            "bytes_saved": before.st_size - compact_bytes,\n            "size_fraction_of_source": compact_bytes / before.st_size,\n            "original_json_bytes_preserved": True,\n            "old_models_included": False,\n            "reference_archive_bytes_preserved": reference_metadata is not None,\n            "required_dataset_visibility": "private",\n            **verification,\n        }\n        # Path.rename refuses an existing target on Windows. The prior existence\n        # checks additionally guard ordinary invocations on other platforms.\n        if destination.exists():\n            raise FileExistsError("Output appeared during packaging; refusing to overwrite it")\n        temporary.rename(destination)\n        return report\n    finally:\n        if temporary is not None:\n            temporary.unlink(missing_ok=True)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--source", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--reference-analysis", type=Path, required=True)\n    parser.add_argument("--report", type=Path)\n    args = parser.parse_args()\n    report_path = args.report or args.output.with_suffix(".report.json")\n    if report_path.exists() or report_path.is_symlink():\n        raise FileExistsError("Packaging report already exists; choose a new path")\n    if report_path.resolve() in {args.source.resolve(), args.output.resolve(), args.reference_analysis.resolve()}:\n        raise ValueError("Report must be separate from source and output archives")\n    report = package_inputs(args.source, args.output, reference_analysis=args.reference_analysis)\n    report_path.parent.mkdir(parents=True, exist_ok=True)\n    temporary: Path | None = None\n    try:\n        with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=report_path.parent,\n                                         prefix=".ver29_report_", suffix=".tmp", delete=False) as writer:\n            temporary = Path(writer.name)\n            json.dump(report, writer, ensure_ascii=False, indent=2)\n            writer.write("\\n")\n        if report_path.exists():\n            raise FileExistsError("Packaging report appeared during packaging")\n        temporary.rename(report_path)\n    finally:\n        if temporary is not None:\n            temporary.unlink(missing_ok=True)\n    print(json.dumps(report, ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'reference.py': '"""Import the pinned completed ver28 prediction reference, never old weights/code."""\nfrom __future__ import annotations\n\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport re\nimport stat\nimport zipfile\n\nimport numpy as np\n\nfrom . import analysis, bootstrap, data\n\nREFERENCE_FORMAT = "mission3_ver28_v1"\nREFERENCE_ARCHIVE = "ver28_analysis_20261013T005822Z_1de5c4ec.zip"\nEXPECTED_MANIFEST_SHA256 = "97d5ea739790354e4cc73c2a0643945dc7ff2595522c7829ef9e012024e3ceaf"\nMAX_FILE_BYTES = 32 * 1024 * 1024\nBASE_FILES = {"run_state.json", "settings.json", "screening_complete.json", "selection.json",\n              "resolved_models.json", "split_manifest.json", "split_indices.npz",\n              "development_predictions.npz", "confirmation_predictions.npz", "confirmation_report.json"}\nPOLICY = "independent sigmoid >= 0.5; ensemble is fixed equal probability mean"\nIDENTITY_POLICY = "independent_logits_greater_than_or_equal_to_zero"\n\n\ndef _json(raw):\n    return json.loads(raw.decode("utf-8-sig"))\n\n\ndef _sha(raw):\n    return hashlib.sha256(raw).hexdigest()\n\n\ndef _required(manifest):\n    names = set(BASE_FILES)\n    for kind in ("control", "kure"):\n        names.update(f"models/{kind}/{name}" for name in\n                     ("training_identity.json", "history.json", "training_result.json"))\n    source = {name for name in manifest["files"] if name.startswith("source_snapshot/")}\n    if not {"source_snapshot/trainer.py", "source_snapshot/runtime.py", "source_snapshot/data.py",\n            "source_snapshot/analysis.py", "source_snapshot/settings.py"}.issubset(source):\n        raise ValueError("Reference manifest omits its source provenance")\n    return names | source\n\n\ndef _verified_contents(read):\n    raw = read("artifact_manifest.json")\n    if _sha(raw) != EXPECTED_MANIFEST_SHA256:\n        raise ValueError("Reference must be the exact audited completed ver28 run")\n    manifest = _json(raw)\n    if manifest.get("format") != REFERENCE_FORMAT or not isinstance(manifest.get("files"), dict):\n        raise ValueError("Reference manifest format is invalid")\n    result = {"artifact_manifest.json": raw}\n    for name in sorted(_required(manifest)):\n        bootstrap._relative(name)\n        metadata = manifest["files"].get(name, {})\n        if type(metadata.get("size")) is not int or not 0 <= metadata["size"] <= MAX_FILE_BYTES:\n            raise ValueError("Reference required artifact missing or unexpectedly large")\n        payload = read(name)\n        if len(payload) != metadata["size"] or _sha(payload) != metadata.get("sha256"):\n            raise ValueError("Reference artifact integrity failed: " + name)\n        result[name] = payload\n    return result\n\n\ndef _read_directory(root):\n    def read(name):\n        path = bootstrap._target(root.resolve(), name)\n        if not path.is_file() or path.stat().st_size > MAX_FILE_BYTES:\n            raise ValueError("Reference required artifact missing or too large")\n        return path.read_bytes()\n    return _verified_contents(read)\n\n\ndef _read_zip(payload):\n    with zipfile.ZipFile(payload) as archive:\n        names = set()\n        for info in archive.infolist():\n            bootstrap._relative(info.orig_filename)\n            key = info.filename.rstrip(\'/\').casefold()\n            if key in names or stat.S_ISLNK(info.external_attr >> 16) or info.flag_bits & 1:\n                raise ValueError("Unsafe reference ZIP entry")\n            names.add(key)\n        matches = [n for n in archive.namelist() if n.endswith(\'/artifact_manifest.json\') or n == \'artifact_manifest.json\']\n        if len(matches) != 1:\n            raise ValueError("Reference ZIP must contain exactly one run manifest")\n        prefix = matches[0][:-len(\'artifact_manifest.json\')]\n        def read(name):\n            info = archive.getinfo(prefix + name)\n            if info.file_size > MAX_FILE_BYTES:\n                raise ValueError("Reference ZIP artifact is unexpectedly large")\n            return archive.read(info)\n        return _verified_contents(read)\n\n\ndef _read_source(path):\n    if path.is_dir():\n        return _read_directory(path)\n    if path.is_file() and path.suffix.lower() == \'.zip\':\n        with zipfile.ZipFile(path) as archive:\n            if \'bundle_manifest.json\' in archive.namelist():\n                manifest = _json(archive.read(\'bundle_manifest.json\'))\n                item = manifest.get(\'reference_analysis\', {})\n                name = item.get(\'path\', \'\')\n                bootstrap._relative(name)\n                info = archive.getinfo(name)\n                if info.file_size > MAX_FILE_BYTES or info.file_size != item.get(\'size\'):\n                    raise ValueError("Bundled reference size mismatch")\n                raw = archive.read(info)\n                if _sha(raw) != item.get(\'sha256\'):\n                    raise ValueError("Bundled reference hash mismatch")\n                return _read_zip(io.BytesIO(raw))\n        return _read_zip(path)\n    raise ValueError("Reference path is not a run folder or ZIP")\n\n\ndef locate(settings):\n    explicit = settings.get(\'reference_run\', \'\')\n    if explicit:\n        root = Path(explicit).expanduser().resolve()\n        if root.is_file() or (root / \'artifact_manifest.json\').is_file():\n            return root\n        roots = [root]\n    else:\n        roots = [Path(settings[\'input_root\']).expanduser()]\n        if settings.get(\'input_zip\'):\n            item = Path(settings[\'input_zip\']).expanduser()\n            if item.is_file():\n                return item.resolve()\n            if item.is_dir():\n                roots.append(item)\n    candidates = set()\n    for root in roots:\n        if not root.is_dir():\n            continue\n        for path in root.rglob(REFERENCE_ARCHIVE):\n            candidates.add(path.resolve())\n        for path in root.rglob(\'mission3_ver29_inputs.zip\'):\n            candidates.add(path.resolve())\n        for marker in root.rglob(\'artifact_manifest.json\'):\n            if marker.stat().st_size <= MAX_FILE_BYTES and _sha(marker.read_bytes()) == EXPECTED_MANIFEST_SHA256:\n                candidates.add(marker.parent.resolve())\n    # Multiple representations of the pinned reference are equivalent, but\n    # every chosen source still undergoes exact manifest/content verification.\n    if not candidates:\n        raise FileNotFoundError("Attach the ver29 input bundle with the audited ver28 reference")\n    return sorted(candidates, key=lambda p: (p.name != REFERENCE_ARCHIVE, str(p)))[0]\n\n\ndef _arrays(contents, name):\n    with np.load(io.BytesIO(contents[name]), allow_pickle=False) as arrays:\n        return {key: arrays[key].copy() for key in arrays.files}\n\n\ndef import_reference(settings, output, texts, targets, indices, split_manifest):\n    """Verify against current original data, then preserve only small evidence."""\n    snapshot = Path(output) / \'reference_snapshot\'\n    if (snapshot / \'artifact_manifest.json\').is_file():\n        contents, source = _read_directory(snapshot), \'self-contained reference_snapshot\'\n    else:\n        path = locate(settings)\n        contents, source = _read_source(path), str(path)\n    state = _json(contents[\'run_state.json\'])\n    config = _json(contents[\'settings.json\'])\n    split = _json(contents[\'split_manifest.json\'])\n    selection = _json(contents[\'selection.json\'])\n    if (state.get(\'format\') != REFERENCE_FORMAT or state.get(\'status\') != \'screening_complete\'\n            or state.get(\'run_mode\') != \'screening\' or state.get(\'submission_ready\') is not False\n            or config.get(\'run_mode\') != \'screening\' or config.get(\'run_validation\') is not False):\n        raise ValueError("Reference must be completed screening without official Validation selection")\n    if (split.get(\'data_digest\') != split_manifest[\'data_digest\']\n            or split.get(\'split_digest\') != split_manifest[\'split_digest\']\n            or split.get(\'label_order\') != list(data.TARGET_SYMPTOMS)\n            or selection.get(\'split_digest\') != split_manifest[\'split_digest\']\n            or selection.get(\'training_rows\') != len(indices[\'fit\'])\n            or selection.get(\'selected_recipe\') != \'mean\' or selection.get(\'members\') != [\'control\', \'kure\']\n            or selection.get(\'decision_rule\') != POLICY\n            or selection.get(\'selected_epochs\') != {\'control\': 3, \'kure\': 2}):\n        raise ValueError("Reference data, split, labels, epochs, or policy differs")\n    if data.dataset_digest(texts, targets) != split[\'data_digest\']:\n        raise ValueError("Reference original data hash mismatch")\n    old_indices = _arrays(contents, \'split_indices.npz\')\n    if set(old_indices) != set(indices) or any(not np.array_equal(old_indices[k], v) for k,v in indices.items()):\n        raise ValueError("Reference row indices differ")\n    specs = _json(contents[\'resolved_models.json\'])\n    identity_digest = hashlib.sha256()\n    for text in texts:\n        raw = text.encode(\'utf-8\')\n        identity_digest.update(len(raw).to_bytes(8, \'little\'))\n        identity_digest.update(raw)\n    # ver28 trainer validates/casts labels to float32 before its identity hash.\n    identity_digest.update(np.asarray(targets, dtype=np.float32).tobytes())\n    for kind in (\'control\', \'kure\'):\n        identity = _json(contents[f\'models/{kind}/training_identity.json\'])\n        history = _json(contents[f\'models/{kind}/history.json\'])\n        result = _json(contents[f\'models/{kind}/training_result.json\'])\n        if (identity.get(\'format\') != \'mission3_ver28_full_encoder_v1\'\n                or identity.get(\'kind\') != kind or identity.get(\'target_symptoms\') != list(data.TARGET_SYMPTOMS)\n                or identity.get(\'decision\') != IDENTITY_POLICY or identity.get(\'data_sha256\') != identity_digest.hexdigest()\n                or identity.get(\'train_indices\') != indices[\'fit\'].tolist()\n                or identity.get(\'dev_indices\') != indices[\'development\'].tolist()\n                or identity.get(\'epochs\') != 3 or result.get(\'status\') != \'complete\'\n                or result.get(\'best_epoch\') != selection[\'selected_epochs\'][kind]\n                or max(history, key=lambda r: r[\'macro_f1\'])[\'epoch\'] != result[\'best_epoch\']\n                or identity.get(\'model_id\') != specs[kind][\'model_id\']\n                or identity.get(\'revision\') != specs[kind][\'revision\']\n                or not re.fullmatch(\'[0-9a-f]{40}\', specs[kind][\'revision\'])):\n            raise ValueError("Reference model provenance or original fit rows differ")\n        for name, value in identity[\'settings\'].items():\n            if settings.get(name) != value:\n                raise ValueError("Reference shared training recipe differs: " + name)\n    development = _arrays(contents, \'development_predictions.npz\')\n    confirmation = _arrays(contents, \'confirmation_predictions.npz\')\n    for name, arrays in ((\'development\', development), (\'confirmation\', confirmation)):\n        if not np.array_equal(arrays[\'indices\'], indices[name]) or not np.array_equal(arrays[\'targets\'], targets[indices[name]]):\n            raise ValueError("Reference predictions do not align with original targets")\n    observed = analysis.ensemble_logits(development[\'control\'], development[\'kure\'])\n    if not np.allclose(observed, development[\'mean\'], rtol=1e-12, atol=1e-12) or not np.array_equal(observed >= 0, development[\'mean\'] >= 0):\n        raise ValueError("Reference is not the fixed equal baseline")\n    for kind in (\'control\', \'kure\', \'mean\'):\n        if analysis.metrics(development[\'targets\'], development[kind]) != selection[\'scores\'][kind]:\n            raise ValueError("Reference development score failed reproduction")\n    confirm_report = _json(contents[\'confirmation_report.json\'])\n    if analysis.metrics(confirmation[\'targets\'], confirmation[\'selected\']) != confirm_report[\'selected\']:\n        raise ValueError("Reference confirmation score failed reproduction")\n    snapshot.mkdir(parents=True, exist_ok=True)\n    for name, raw in contents.items():\n        dest = bootstrap._target(snapshot.resolve(), name)\n        dest.parent.mkdir(parents=True, exist_ok=True)\n        dest.write_bytes(raw)\n    old_provenance = Path(output) / \'reference_provenance.json\'\n    original_source = _json(old_provenance.read_bytes()).get(\'source\', source) if old_provenance.is_file() else source\n    provenance = {\'format\': \'mission3_ver29_reference_v1\', \'source\': original_source,\n        \'reference_manifest_sha256\': EXPECTED_MANIFEST_SHA256, \'verified_artifacts\': len(contents)-1,\n        \'partial_artifact_copy\': True, \'weights_required\': False, \'original_labels_and_indices_verified\': True,\n        \'baseline_epochs\': selection[\'selected_epochs\'], \'baseline_training_rows\': len(indices[\'fit\']),\n        \'baseline_precision\': \'BF16 on T4; ver29 fresh training uses native-supported precision\',\n        \'comparison_limitation\': \'Historical baseline reuses the same split; precision changes mean this is not a pure loss ablation.\',\n        \'source_not_executed\': True}\n    data.write_json(Path(output) / \'reference_provenance.json\', provenance)\n    return {\'development\': analysis._logits(development[\'mean\'], len(indices[\'development\'])),\n            \'confirmation\': analysis._logits(confirmation[\'selected\'], len(indices[\'confirmation\'])),\n            \'selected_epochs\': selection[\'selected_epochs\'], \'specs\': specs, \'provenance\': provenance}\n', 'run_notebook.py': '"""Fresh-process notebook entry point with privacy-safe error diagnostics."""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nimport traceback\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--settings", required=True)\n    parser.add_argument("--result-file", required=True)\n    args = parser.parse_args(argv)\n    result_path = Path(args.result_file)\n    result_path.parent.mkdir(parents=True, exist_ok=True)\n    before = set()\n    try:\n        from . import data, runtime\n        from .build_notebook import source_files\n        root = Path(__file__).resolve().parent\n        sources = {name: (root / name).read_text(encoding="utf-8") for name in source_files()}\n        settings = json.loads(Path(args.settings).read_text(encoding="utf-8-sig"))\n        before = set(Path(settings["output_dir"]).glob("*/run_state.json"))\n        result = runtime.run(settings, runtime_sources=sources)\n        data.write_json(result_path, result)\n        return 0\n    except Exception as error:\n        frames = [{"file": Path(frame.f_code.co_filename).name, "line": line, "function": frame.f_code.co_name}\n                  for frame, line in traceback.walk_tb(error.__traceback__)]\n        # No raw exception values, transcripts, or patient metadata are logged.\n        report = {"status": "failed", "submission_ready": False,\n                  "error_type": type(error).__name__, "frames": frames[-15:]}\n        try:\n            settings = json.loads(Path(args.settings).read_text(encoding="utf-8-sig"))\n            runs = set(Path(settings["output_dir"]).glob("*/run_state.json")) - before\n            if len(runs) == 1:\n                report["run_dir"] = str(next(iter(runs)).parent)\n        except (OSError, ValueError, KeyError):\n            pass\n        result_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")\n        print(json.dumps(report, ensure_ascii=False), flush=True)\n        return 1\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'runtime.py': '"""Kaggle screening, immutable selection, resumable refit and final evaluation."""\nfrom __future__ import annotations\n\nfrom datetime import datetime, timezone\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\nimport time\nimport traceback\nimport uuid\n\nimport numpy as np\n\nfrom . import analysis, bootstrap, data, reference, trainer\nfrom .budget import Budget, BudgetExceeded\nfrom .settings import recipe_settings, validate_settings\n\nFORMAT = "mission3_ver29_v1"\nEXPECTED_TRAINING_SAMPLES = 29200\nEXPECTED_VALIDATION_SAMPLES = 3640\nEXPECTED_DATA_DIGEST = "71b751de9931f10a6d5e2b17d38bf36d0f986baef82e47f7d98159a30348aa6f"\nEXPECTED_SPLIT_DIGEST = "deac9096788901eb8652169e8f97c98f35b21b08f307f73751a253646b75efa7"\nCORE_SOURCES = {"trainer.py", "runtime.py", "data.py", "analysis.py", "reference.py", "settings.py", "bootstrap.py", "budget.py"}\nNEW_MEMBERS = ("klue_asl", "qwen_embedding")\nRECIPES = {"baseline": ["control", "kure"], "klue_asl": ["klue_asl"],\n           "qwen_embedding": ["qwen_embedding"], "new_pair": list(NEW_MEMBERS),\n           "all_four": ["control", "kure", *NEW_MEMBERS]}\nINFERENCE_ENTRYPOINT = (\n    \'"""Competition entrypoint; keep source_snapshot and model assets beside this file."""\\n\'\n    \'from source_snapshot.inference import main\\n\\n\'\n    \'if __name__ == "__main__":\\n    main()\\n\')\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8-sig"))\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef save_npz(path, **arrays):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(".tmp")\n    with temporary.open("wb") as stream:\n        np.savez_compressed(stream, **arrays)\n    temporary.replace(path)\n\n\ndef find_run(path):\n    root = Path(path).expanduser().resolve()\n    if (root / "run_state.json").is_file() and read_json(root / "run_state.json").get("format") == FORMAT:\n        return root\n    matches = [p for p in root.rglob("run_state.json") if read_json(p).get("format") == FORMAT] if root.is_dir() else []\n    if len(matches) != 1:\n        raise ValueError("Specify the exact ver29 run folder containing run_state.json")\n    return matches[0].parent\n\n\ndef write_manifest(root):\n    files = {}\n    for path in sorted(root.rglob("*")):\n        if path.is_file() and path != root / "artifact_manifest.json" and path.suffix != ".tmp":\n            if path.is_symlink():\n                raise ValueError("Run artifacts must not contain symlinks")\n            files[path.relative_to(root).as_posix()] = {"size": path.stat().st_size, "sha256": file_hash(path)}\n    data.write_json(root / "artifact_manifest.json", {"format": FORMAT, "files": files})\n\n\ndef verify_run(root):\n    manifest = read_json(root / "artifact_manifest.json")\n    if manifest.get("format") != FORMAT or not isinstance(manifest.get("files"), dict):\n        raise ValueError("Source run manifest is not ver29")\n    if any(path.is_symlink() for path in root.rglob("*")):\n        raise ValueError("Source run must not contain symlinks")\n    actual = {path.relative_to(root).as_posix() for path in root.rglob("*")\n              if path.is_file() and path != root / "artifact_manifest.json" and path.suffix != ".tmp"}\n    if actual != set(manifest["files"]):\n        raise ValueError("Source run manifest has missing or unexpected files")\n    for name, metadata in manifest["files"].items():\n        path = bootstrap._target(root, name)\n        if not path.is_file() or path.stat().st_size != metadata["size"] or file_hash(path) != metadata["sha256"]:\n            raise ValueError("Source run artifact integrity failed: " + name)\n    for name in ("run_state.json", "settings.json"):\n        if name not in manifest["files"]:\n            raise ValueError("Source run manifest omits required metadata")\n    state = read_json(root / "run_state.json")\n    if state.get("format") != FORMAT:\n        raise ValueError("Source run state is not ver29")\n    if state["status"] in ("screening_complete", "complete"):\n        for name in ("split_manifest.json", "selection.json", "resolved_models.json"):\n            if name not in manifest["files"]:\n                raise ValueError("Completed source run omits selection metadata")\n    return state\n\n\ndef resolve_inputs(settings):\n    if settings["train_dir"] or settings["validation_dir"]:\n        return bootstrap.prepare_data("", settings["work_dir"], settings["train_dir"], settings["validation_dir"])\n    explicit = Path(settings["input_zip"]).expanduser() if settings["input_zip"] else None\n    if explicit and explicit.is_file():\n        return bootstrap.prepare_data(explicit, settings["work_dir"])\n    search_root = explicit or Path(settings["input_root"])\n    if not search_root.is_dir():\n        raise FileNotFoundError("Kaggle input not found; attach the private ver29 input Dataset")\n    archives = list(search_root.rglob("mission3_ver29_inputs.zip"))\n    if len(archives) == 1:\n        return bootstrap.prepare_data(archives[0], settings["work_dir"])\n    if len(archives) > 1:\n        raise ValueError("Multiple ver29 input ZIPs found; set INPUT_ZIP explicitly")\n    # Kaggle may unzip Dataset uploads automatically; verify those exact bytes too.\n    candidates = []\n    for marker in search_root.rglob("bundle_manifest.json"):\n        manifest = read_json(marker)\n        base = marker.parent\n        train = base / bootstrap.TRAIN_PREFIX\n        valid = base / bootstrap.VALID_PREFIX\n        if manifest.get("format") == "mission3_ver5_inputs_v1" and train.is_dir() and valid.is_dir():\n            candidates.append((base, train, valid, manifest))\n    if len(candidates) != 1:\n        raise ValueError("Attach exactly one ver29 input Dataset, or set INPUT_ZIP to its extracted folder")\n    base, train, valid, manifest = candidates[0]\n    selected = [row for row in manifest["files"] if row["path"].startswith((bootstrap.TRAIN_PREFIX, bootstrap.VALID_PREFIX))]\n    expected = {row["path"] for row in selected}\n    if len(expected) != len(selected):\n        raise ValueError("Duplicate input manifest rows")\n    actual = {path.relative_to(base).as_posix() for root in (train, valid) for path in root.rglob("*.json")}\n    if actual != expected:\n        raise ValueError("Extracted Dataset has missing or extra label JSON files")\n    for row in selected:\n        path = bootstrap._target(base.resolve(), row["path"])\n        if path.stat().st_size != row["size"] or file_hash(path) != row["sha256"]:\n            raise ValueError("Extracted Dataset content hash mismatch")\n    return train, valid\n\n\ndef resolve_spec(kind, spec, settings, budget, output):\n    from huggingface_hub import HfApi, snapshot_download\n    budget.check("pretrained model download", training=True)\n    spec = dict(spec)\n    revision = spec["revision"]\n    if not re.fullmatch(r"[0-9a-f]{40}", revision):\n        revision = HfApi().model_info(spec["model_id"], revision=revision).sha\n    if not re.fullmatch(r"[0-9a-f]{40}", revision or ""):\n        raise ValueError("Pretrained revision was not resolved to an immutable commit")\n    # Persist the resolved identity even if a long download is interrupted.\n    spec.update(kind=kind, revision=revision)\n    specs = read_json(output / "resolved_models.json") if (output / "resolved_models.json").exists() else {}\n    specs[kind] = {key: value for key, value in spec.items() if key != "snapshot_path"}\n    data.write_json(output / "resolved_models.json", specs)\n    snapshot = snapshot_download(spec["model_id"], revision=revision, cache_dir=settings["cache_dir"],\n        allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.jinja", "LICENSE*", "README.md"],\n        ignore_patterns=["onnx/*", "openvino/*", "original/*"])\n    spec["snapshot_path"] = str(snapshot)\n    notices = output / "licenses" / kind\n    notices.mkdir(parents=True, exist_ok=True)\n    for path in Path(snapshot).iterdir():\n        if path.is_file() and (path.name.startswith("LICENSE") or path.name == "README.md"):\n            shutil.copyfile(path, notices / path.name)\n    pooling = Path(snapshot) / "1_Pooling" / "config.json"\n    if pooling.is_file():\n        shutil.copyfile(pooling, notices / "upstream_pooling_config.json")\n    data.write_json(notices / "provenance.json", specs[kind])\n    budget.check("pretrained model ready", training=True)\n    return spec\n\n\ndef recipe_logits(logits):\n    new_pair = analysis.ensemble_logits(logits["klue_asl"], logits["qwen_embedding"])\n    return {"baseline": logits["baseline"], "klue_asl": logits["klue_asl"],\n            "qwen_embedding": logits["qwen_embedding"], "new_pair": new_pair,\n            "all_four": analysis.ensemble_logits(logits["baseline"], new_pair)}\n\n\ndef choose_candidate(targets, logits):\n    candidates = recipe_logits(logits)\n    scores = {name: analysis.metrics(targets, values) for name, values in candidates.items()}\n    # Prefer a simpler existing control for exact-score ties.\n    order = tuple(RECIPES)\n    winner = max(order, key=lambda name: scores[name]["macro_f1"])\n    comparison = analysis.paired_bootstrap_ci(targets, logits["baseline"], candidates[winner])\n    delta = comparison["macro_f1_delta"]\n    return {"selected_recipe": winner, "members": RECIPES[winner],\n            "scores": scores, "paired_comparison": comparison,\n            "promising": delta["estimate"] >= .005 and delta["lower"] > 0,\n            "decision_rule": "independent sigmoid >= 0.5; ensemble is fixed equal probability mean",\n            "limitations": "Same-fit historical baseline; BF16 versus native-supported precision is not a pure loss ablation. Development and confirmation have prior experiment exposure; "\n                            "bootstrap excludes repeated selection and training-seed variation. Not an official 0.7 result."}, candidates\n\n\ndef _release():\n    import torch\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef _predict(root, kind, texts, ids, device, budget):\n    batch_size = read_json(root / "models" / kind / "batch_benchmark.json")["selected_batch_size"]\n    values = trainer.predict_checkpoint(root / "models" / kind / "best_model", texts, ids,\n        device=device, batch_size=batch_size, budget=budget)\n    _release()\n    return analysis._logits(values, len(texts) if ids is None else len(ids))\n\n\ndef _finish(output, state, status, started, **extra):\n    if status in ("screening_complete", "complete"):\n        state.pop("pause_stage", None)\n        state.pop("error_type", None)\n    state.update(status=status, elapsed_seconds=time.monotonic() - started, **extra)\n    data.write_json(output / "run_state.json", state)\n    write_manifest(output)\n    result = {"run_dir": str(output), "status": status, "submission_ready": status == "complete"}\n    print(json.dumps(result, ensure_ascii=False), flush=True)\n    return result\n\n\ndef _verify_source_recipe(source, settings, split_manifest):\n    if recipe_settings(read_json(source / "settings.json")) != recipe_settings(settings):\n        raise ValueError("Source and current experiment recipes differ")\n    if ((source / "split_manifest.json").exists()\n            and read_json(source / "split_manifest.json")["split_digest"] != split_manifest["split_digest"]):\n        raise ValueError("Source and current Training split differ")\n\n\ndef _verify_reload(output, kind, texts, device, budget):\n    root = output / "models" / kind\n    with np.load(root / "reload_reference.npz", allow_pickle=False) as reference:\n        indices, expected = reference["indices"], reference["logits"]\n    observed = _predict(output, kind, texts, indices, device, budget)\n    if not (np.allclose(observed, expected, rtol=1e-4, atol=1e-5)\n            and np.array_equal(observed >= 0, expected >= 0)):\n        raise ValueError("Saved model does not reproduce the live model reference")\n    data.write_json(root / "reload_verification.json", {"passed": True, "samples": len(indices),\n        "max_abs_logit_difference": float(np.max(np.abs(observed - expected))), "decisions_identical": True})\n\n\ndef run(settings, runtime_sources=None):\n    import torch\n    settings = validate_settings(settings)\n    started = settings.get("invocation_started_monotonic", time.monotonic())\n    deadline = settings.get("deadline_monotonic", started + settings["time_budget_minutes"] * 60)\n    budget = Budget(deadline, settings["finalization_reserve_seconds"], settings["budget_safety_factor"])\n    output = Path(settings["output_dir"]).expanduser().resolve() / (\n        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ_") + uuid.uuid4().hex[:8])\n    output.mkdir(parents=True, exist_ok=False)\n    state = {"format": FORMAT, "status": "starting", "run_mode": settings["run_mode"],\n             "candidate": settings["candidate"], "submission_ready": False, "completed_models": {}}\n    sources = runtime_sources or {}\n    try:\n        device = "cuda" if torch.cuda.is_available() else "cpu"\n        if device == "cpu" and not settings["allow_cpu"]:\n            raise RuntimeError("CUDA GPU required; choose Kaggle Accelerator GPU")\n        resume = None\n        if settings["resume_run"]:\n            resume = find_run(settings["resume_run"])\n            old_state = verify_run(resume)\n            if old_state["status"] != "paused" or old_state["run_mode"] != settings["run_mode"]:\n                raise ValueError("RESUME_RUN must be a paused ver29 run of the same mode")\n            for name in CORE_SOURCES & sources.keys():\n                if (resume / "source_snapshot" / name).read_text(encoding="utf-8") != sources[name]:\n                    raise ValueError("Resume source code differs: " + name)\n            shutil.copytree(resume, output, dirs_exist_ok=True, ignore=shutil.ignore_patterns("*.tmp"))\n            state = {**old_state, "status": "resuming", "resumed_from": str(resume)}\n        data.write_json(output / "settings.json", settings)\n        data.write_json(output / "run_state.json", state)\n        for name, content in sources.items():\n            path = output / "source_snapshot" / name\n            path.parent.mkdir(parents=True, exist_ok=True)\n            path.write_text(content, encoding="utf-8")\n        packages = {}\n        for name in ("torch", "transformers", "numpy", "scikit-learn", "scipy", "accelerate"):\n            try:\n                packages[name] = importlib.metadata.version(name)\n            except importlib.metadata.PackageNotFoundError:\n                packages[name] = None\n        data.write_json(output / "environment.json", {"packages": packages, "device": device,\n            "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,\n            "gpu_count": torch.cuda.device_count(), "used_gpu_count": int(device == "cuda")})\n        budget.check("data preparation", training=True)\n        train_dir, valid_dir = resolve_inputs(settings)\n        paths, texts, targets = data.load_dataset(train_dir, include_targets=True)\n        if len(texts) != EXPECTED_TRAINING_SAMPLES or data.dataset_digest(texts, targets) != EXPECTED_DATA_DIGEST:\n            raise ValueError("Original Training count or text/target digest does not match the verified dataset")\n        fit, development, confirmation, split_manifest = data.three_way_split(texts, targets, seed=settings["seed"])\n        if split_manifest["split_digest"] != EXPECTED_SPLIT_DIGEST:\n            raise ValueError("Internal split differs from ver25-ver27")\n        if resume:\n            _verify_source_recipe(resume, settings, split_manifest)\n        data.write_json(output / "split_manifest.json", split_manifest)\n        save_npz(output / "split_indices.npz", fit=fit, development=development, confirmation=confirmation)\n        specs = read_json(Path(__file__).with_name("model_specs.json"))\n        if (output / "resolved_models.json").exists():\n            specs.update(read_json(output / "resolved_models.json"))\n        selection = None\n        if settings["run_mode"] == "full":\n            if resume and (output / "selection.json").is_file():\n                selection = read_json(output / "selection.json")\n            else:\n                source_path = settings["source_run"] or (read_json(resume / "settings.json")["source_run"] if resume else "")\n                if not source_path:\n                    raise ValueError("Full preparation has no frozen selection yet; attach completed screening and set SOURCE_RUN")\n                source = find_run(source_path)\n                source_state = verify_run(source)\n                if source_state["status"] != "screening_complete" or source_state["run_mode"] != "screening":\n                    raise ValueError("SOURCE_RUN must be completed ver29 screening")\n                _verify_source_recipe(source, settings, split_manifest)\n                for name in CORE_SOURCES & sources.keys():\n                    if (source / "source_snapshot" / name).read_text(encoding="utf-8") != sources[name]:\n                        raise ValueError("Full source implementation differs from screening: " + name)\n                selection = read_json(source / "selection.json")\n                specs.update(read_json(source / "resolved_models.json"))\n                shutil.copytree(source / "reference_snapshot", output / "reference_snapshot", dirs_exist_ok=True)\n                shutil.copyfile(source / "reference_provenance.json", output / "reference_provenance.json")\n                data.write_json(output / "selection.json", selection)\n                data.write_json(output / "source_screening.json", {"source_run": str(source),\n                    "artifact_manifest_sha256": file_hash(source / "artifact_manifest.json")})\n            members = selection["members"]\n            if members != RECIPES.get(selection.get("selected_recipe")):\n                raise ValueError("Invalid selected model members")\n            data.write_json(output / "resolved_models.json", {kind: specs[kind] for kind in members})\n        else:\n            members = list(NEW_MEMBERS)\n        reference_values = reference.import_reference(settings, output, texts, targets,\n            {"fit": fit, "development": development, "confirmation": confirmation}, split_manifest)\n        if settings["run_mode"] == "screening":\n            specs.update(reference_values["specs"])\n            data.write_json(output / "resolved_models.json", specs)\n        for kind in members:\n            if kind in state["completed_models"]:\n                continue\n            budget.check("model training", training=True)\n            spec = resolve_spec(kind, specs[kind], settings, budget, output)\n            model_dir = output / "models" / kind\n            epochs = settings["max_epochs"] if selection is None else selection["selected_epochs"][kind]\n            if type(epochs) is not int or not 1 <= epochs <= settings["max_epochs"]:\n                raise ValueError("Invalid frozen selected epoch")\n            print(f"Training {kind}: rows={len(fit) if selection is None else len(texts)}, epochs={epochs}", flush=True)\n            result = trainer.train_model(kind, spec, settings, texts, targets,\n                fit if selection is None else np.arange(len(texts)), development if selection is None else None,\n                model_dir, budget, epochs, resume=(model_dir / "training_identity.json").is_file())\n            _release()\n            if result["status"] == "paused":\n                return _finish(output, state, "paused", started, pause_stage="training " + kind)\n            if result["status"] != "complete":\n                raise ValueError("Trainer returned an unexpected status")\n            state["completed_models"][kind] = result\n            data.write_json(output / "run_state.json", state)\n        for kind in members:\n            if not (output / "models" / kind / "reload_verification.json").exists():\n                _verify_reload(output, kind, texts, device, budget)\n        if settings["run_mode"] == "screening":\n            budget.check("development candidate comparison", training=True)\n            logits = {"baseline": reference_values["development"]}\n            for kind in members:\n                epoch = state["completed_models"][kind]["best_epoch"]\n                logits[kind] = np.load(output / "models" / kind / f"epoch_{epoch}_development_logits.npy", allow_pickle=False)\n            candidates = recipe_logits(logits)\n            if (output / "selection.json").is_file():\n                selection = read_json(output / "selection.json")\n                if selection.get("members") != RECIPES.get(selection.get("selected_recipe")):\n                    raise ValueError("Invalid frozen screening selection")\n            else:\n                selection, candidates = choose_candidate(targets[development], logits)\n                epochs = {**reference_values[\'selected_epochs\'],\n                    **{kind: int(state[\'completed_models\'][kind][\'best_epoch\']) for kind in members}}\n                selection.update(format=FORMAT, candidate=settings["candidate"], selected_epochs=epochs,\n                    protocols={kind: trainer.model_protocol(kind) for kind in RECIPES[\'all_four\']},\n                    split_digest=split_manifest["split_digest"], training_rows=len(fit),\n                    reference_manifest_sha256=reference.EXPECTED_MANIFEST_SHA256,\n                    confirmation_policy="Report only; never revise this frozen selection. Already exposed in previous experiments.")\n            # Freeze BEFORE looking at confirmation targets/predictions.\n            data.write_json(output / "selection.json", selection)\n            save_npz(output / "development_predictions.npz", targets=targets[development], indices=development,\n                     **candidates)\n            data.write_json(output / "paired_errors.json", analysis.paired_error_changes(\n                targets[development], logits["baseline"], candidates[selection["selected_recipe"]]))\n            budget.check("confirmation report", training=True)\n            # Only new models are inferred; the historical baseline remains\n            # the exact frozen pair from its original confirmation evaluation.\n            confirm = {"baseline": reference_values["confirmation"]}\n            for kind in NEW_MEMBERS:\n                confirm[kind] = _predict(output, kind, texts, confirmation, device, budget)\n            chosen = recipe_logits(confirm)[selection["selected_recipe"]]\n            save_npz(output / "confirmation_predictions.npz", targets=targets[confirmation],\n                     indices=confirmation, selected=chosen, **confirm)\n            data.write_json(output / "confirmation_report.json", {\n                "selected": analysis.metrics(targets[confirmation], chosen),\n                "baseline": analysis.metrics(targets[confirmation], confirm["baseline"]),\n                "paired_comparison": analysis.paired_bootstrap_ci(targets[confirmation], confirm["baseline"], chosen),\n                "selection_changed": False, "prior_exposure": "Used in ver25; not an untouched test set."})\n            data.write_json(output / "screening_complete.json", {"format": FORMAT, "submission_ready": False,\n                "selection": selection["selected_recipe"], "next_action": "Review results, then run full separately."})\n            return _finish(output, state, "screening_complete", started)\n        # Frozen full-training recipe; official Validation cannot change it.\n        budget.check("official Validation inference", training=True)\n        valid_paths, valid_texts, valid_targets = data.load_dataset(valid_dir, include_targets=True)\n        if len(valid_texts) != EXPECTED_VALIDATION_SAMPLES:\n            raise ValueError("Official Validation sample count does not match the original dataset")\n        predictions = {kind: _predict(output, kind, valid_texts, None, device, budget) for kind in members}\n        final_logits = analysis.equal_mean_logits([predictions[kind] for kind in members])\n        data.write_predictions(valid_paths, final_logits >= 0, output / "validation_predictions.csv")\n        save_npz(output / "validation_predictions.npz", targets=valid_targets, logits=final_logits,\n                 filenames=np.asarray([path.name for path in valid_paths]))\n        scores = analysis.metrics(valid_targets, final_logits)\n        data.write_json(output / "validation_metrics.json", scores)\n        data.write_json(output / "submission_manifest.json", {"format": FORMAT, "label_order": list(data.TARGET_SYMPTOMS),\n            "members": {kind: "models/" + kind + "/best_model" for kind in members},\n            "ensemble": "equal_probability_mean" if len(members) > 1 else "single",\n            "decision_threshold": .5, "selected_epochs": selection["selected_epochs"],\n            "training_samples": len(texts), "validation_macro_f1": scores["macro_f1"],\n            "above_0_7": scores["macro_f1"] > .7, "selection_frozen_before_official_validation": True})\n        data.write_json(output / "run_complete.json", {"format": FORMAT, "submission_ready": True,\n            "macro_f1": scores["macro_f1"], "above_0_7": scores["macro_f1"] > .7})\n        (output / "inference.py").write_text(INFERENCE_ENTRYPOINT, encoding="utf-8")\n        (output / "INFERENCE_README.md").write_text(\n            "# ver29 inference\\n\\nKeep this complete run directory, including models, licenses and source_snapshot.\\n"\n            "From this directory, install source_snapshot/requirements.txt in a compatible PyTorch environment, then run:\\n\\n"\n            "```bash\\npython inference.py --audio_dir /path/to/wavs --label_dir /path/to/label_jsons "\n            "--ckpt_path ./submission_manifest.json --output ./outputs/mission3.csv\\n```\\n\\n"\n            "ckpt_path accepts submission_manifest.json or the complete run folder. Audio is accepted but never read.\\n"\n            "The module interface also works: python -m source_snapshot.inference --run-dir . --label-dir /path/to/label_jsons --output predictions.csv\\n"\n            "CUDA is selected when available; use --device cpu to force CPU. Inference reads only utterance text, never symptom labels.\\n"\n            "The models and tokenizers load locally without Hugging Face access.\\n", encoding="utf-8")\n        return _finish(output, state, "complete", started, submission_ready=True)\n    except BudgetExceeded as error:\n        return _finish(output, state, "paused", started, pause_stage=error.stage)\n    except Exception as error:\n        # Raw exception values may contain source texts; retain code positions only.\n        frames = [{"file": Path(frame.f_code.co_filename).name, "line": line,\n                   "function": frame.f_code.co_name} for frame, line in traceback.walk_tb(error.__traceback__)]\n        data.write_json(output / "error.json", {"error_type": type(error).__name__, "frames": frames[-15:]})\n        for name in ("run_complete.json", "screening_complete.json"):\n            marker = output / name\n            if marker.is_file():\n                marker.unlink()\n        _finish(output, state, "failed", started, error_type=type(error).__name__)\n        raise\n', 'settings.py': '"""Explicit, reproducible experiment settings; paths never select a recipe."""\nfrom __future__ import annotations\n\nimport math\n\n\ndef default_settings():\n    return {\n        "version": "ver29", "run_mode": "screening", "candidate": "qwen_embedding",\n        "reference_run": "", "source_run": "", "resume_run": "", "input_zip": "",\n        "input_root": "/kaggle/input", "train_dir": "", "validation_dir": "",\n        "output_dir": "/kaggle/working/results/mission3/ver29",\n        "work_dir": "/kaggle/temp/mission3_ver29", "cache_dir": "/kaggle/temp/hf_ver29",\n        "run_validation": False, "seed": 42, "max_epochs": 3,\n        "kure_max_length": 1024, "control_max_length": 512, "klue_asl_max_length": 512,\n        "qwen_embedding_max_length": 1024,\n        "kure_lr": 2e-5, "control_lr": 2e-5, "klue_asl_lr": 2e-5, "qwen_embedding_lr": 2e-5,\n        "head_lr": 1e-4, "weight_decay": .01, "warmup_ratio": .05,\n        "effective_batch_size": 16, "batch_candidates": [1, 2, 4, 8, 16],\n        "benchmark_examples": 64, "checkpoint_every_updates": 100, "log_every": 50,\n        "gradient_clip": 1., "dropout": .1, "allow_cpu": False,\n        "time_budget_minutes": 480, "finalization_reserve_seconds": 1500,\n        "budget_safety_factor": 1.35, "bootstrap_iterations": 500,\n    }\n\n\nRUNTIME_KEYS = {"reference_run", "source_run", "resume_run", "input_zip", "input_root", "train_dir", "validation_dir",\n                "output_dir", "work_dir", "cache_dir", "run_mode", "run_validation",\n                "time_budget_minutes", "finalization_reserve_seconds", "budget_safety_factor",\n                "deadline_monotonic", "invocation_started_monotonic", "log_every", "allow_cpu"}\n\n\ndef recipe_settings(settings):\n    return {key: value for key, value in settings.items() if key not in RUNTIME_KEYS}\n\n\ndef validate_settings(settings):\n    defaults = default_settings()\n    if not isinstance(settings, dict) or set(settings) - set(defaults) - {\n            "deadline_monotonic", "invocation_started_monotonic"}:\n        raise ValueError("Unknown ver29 settings")\n    merged = {**defaults, **settings}\n    for key, value in defaults.items():\n        if type(merged[key]) is not type(value) and not (\n                type(value) is float and type(merged[key]) in (int, float)):\n            raise ValueError("Invalid setting type: " + key)\n    if merged["run_mode"] not in ("screening", "full"):\n        raise ValueError("run_mode must be screening or full")\n    if merged["candidate"] != "qwen_embedding":\n        raise ValueError("ver29 uses the fixed klue_asl and qwen_embedding screening candidates")\n    if merged["run_validation"] != (merged["run_mode"] == "full"):\n        raise ValueError("Official Validation is evaluated only in full mode")\n    if merged["run_mode"] == "screening" and merged["source_run"]:\n        raise ValueError("screening must not use a full-training source_run")\n    if merged["run_mode"] == "full" and not (merged["source_run"] or merged["resume_run"]):\n        raise ValueError("full requires a completed screening source_run or a full resume_run")\n    for key, value in defaults.items():\n        if key not in RUNTIME_KEYS and merged[key] != value:\n            raise ValueError("Pinned experiment setting changed: " + key)\n    if (not 30 <= merged["time_budget_minutes"] <= 480\n            or not 1200 <= merged["finalization_reserve_seconds"] <= 1800\n            or merged["finalization_reserve_seconds"] >= merged["time_budget_minutes"] * 60\n            or not math.isfinite(merged["budget_safety_factor"])\n            or merged["budget_safety_factor"] < 1.25 or merged["log_every"] < 1):\n        raise ValueError("Invalid time budget or save reserve")\n    for key in ("deadline_monotonic", "invocation_started_monotonic"):\n        if key in merged and (type(merged[key]) not in (int, float)\n                              or not math.isfinite(merged[key]) or merged[key] < 0):\n            raise ValueError("Invalid invocation timestamp")\n    return merged\n', 'trainer.py': '"""Supervised encoder training with measured batches and exact boundary resume.\n\nAll nine predictions use logits >= 0. Model assets reload entirely offline.\nOnly the supplied Training indices contribute to loss or positive weights.\n"""\nfrom __future__ import annotations\n\nfrom contextlib import nullcontext\nimport gc\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport re\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom transformers import AutoConfig, AutoModel, AutoTokenizer, get_linear_schedule_with_warmup\n\nfrom .budget import BudgetExceeded\nfrom .data import TARGET_SYMPTOMS\n\nFORMAT = "mission3_ver29_full_encoder_v1"\nKINDS = ("kure", "control", "klue_asl", "qwen_embedding")\nDECISION = "independent_logits_greater_than_or_equal_to_zero"\nQWEN_INSTRUCTION = "Identify the patient\'s symptoms from this Korean emergency call transcript."\n\n\ndef model_protocol(kind):\n    """Return the immutable objective and text recipe for one candidate."""\n    if kind not in KINDS:\n        raise ValueError("Unsupported model kind")\n    return {\n        "loss": {"name": "weighted_asymmetric_focusing" if kind == "klue_asl" else "weighted_bce",\n                 "gamma_positive": 0., "gamma_negative": 2. if kind == "klue_asl" else 0.,\n                 "probability_clip": 0., "differentiate_focal_factor": True,\n                 "positive_weights": "clip(sqrt(fit_negative/max(fit_positive,1)),1,5)",\n                 "reduction": "mean_over_examples_and_nine_labels"},\n        "preprocessing": {"format": "fixed_instruction_query" if kind == "qwen_embedding" else "plain_text",\n                          "instruction": QWEN_INSTRUCTION if kind == "qwen_embedding" else "",\n                          "template": "Instruct: {instruction}\\nQuery: {text}" if kind == "qwen_embedding" else "{text}",\n                          "truncation": "right", "padding_side": "right"},\n        "pooling": {"kure": "cls", "control": "masked_mean", "klue_asl": "masked_mean",\n                    "qwen_embedding": "last_nonpadding"}[kind],\n        "precision_policy": "native_bfloat16_else_float16_on_cuda",\n    }\n\n\ndef prepare_texts(kind, texts):\n    """Apply the same label-independent input format at training and inference."""\n    protocol = model_protocol(kind)["preprocessing"]\n    if not len(texts) or any(not isinstance(text, str) for text in texts):\n        raise ValueError("Texts must be a nonempty sequence of strings")\n    return [protocol["template"].format(instruction=protocol["instruction"], text=text) for text in texts]\n\n\ndef classification_loss(logits, labels, weights, kind):\n    """Stable ASL variant with no clipping and a differentiated focusing factor."""\n    if kind not in KINDS:\n        raise ValueError("Unsupported model kind")\n    logits = logits.float()\n    if kind != "klue_asl":\n        return F.binary_cross_entropy_with_logits(logits, labels, pos_weight=weights)\n    positive = labels * weights * F.softplus(-logits)\n    negative = (1. - labels) * logits.sigmoid().square() * F.softplus(logits)\n    return (positive + negative).mean()\n\n\ndef _default_maximum(kind):\n    return 512 if kind in ("control", "klue_asl") else 1024\n\n\ndef _get(value, name, default=None):\n    return value.get(name, default) if isinstance(value, dict) else getattr(value, name, default)\n\n\ndef seed_everything(seed):\n    random.seed(int(seed))\n    np.random.seed(int(seed))\n    torch.manual_seed(int(seed))\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(int(seed))\n\n\ndef _targets(values, rows=None):\n    values = np.asarray(values)\n    if (values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS)\n            or (rows is not None and len(values) != rows)\n            or not np.isin(values, (0, 1)).all()):\n        raise ValueError("Targets must be a binary (samples, 9) array")\n    return values.astype(np.float32, copy=False)\n\n\ndef positive_weights(fit_targets):\n    values = _targets(fit_targets)\n    if not len(values):\n        raise ValueError("Positive weights need at least one Training row")\n    positive = values.sum(0, dtype=np.float64)\n    return torch.tensor(np.clip(np.sqrt((len(values) - positive) / np.maximum(positive, 1)), 1, 5),\n                        dtype=torch.float32)\n\n\ndef _indices(values, n):\n    result = np.asarray(values)\n    if (result.ndim != 1 or not len(result) or not np.issubdtype(result.dtype, np.integer)\n            or (result < 0).any() or (result >= n).any() or len(np.unique(result)) != len(result)):\n        raise ValueError("Indices must be nonempty unique valid integers")\n    return result.astype(np.int64)\n\n\ndef _amp_dtype(device):\n    if torch.device(device).type != "cuda":\n        return None\n    with torch.cuda.device(device):\n        supported = torch.cuda.is_bf16_supported(including_emulation=False)\n    return torch.bfloat16 if supported else torch.float16\n\n\ndef _autocast(device):\n    dtype = _amp_dtype(device)\n    return torch.autocast("cuda", dtype=dtype) if dtype is not None else nullcontext()\n\n\ndef _sync(device):\n    if torch.device(device).type == "cuda":\n        torch.cuda.synchronize(device)\n\n\nclass EncoderClassifier(nn.Module):\n    def __init__(self, encoder, kind, dropout=.1):\n        super().__init__()\n        if kind not in KINDS:\n            raise ValueError("Unsupported model kind")\n        expected = {"kure": ("xlm-roberta", "roberta"), "control": ("bert", "roberta"),\n                    "klue_asl": ("bert", "roberta"),\n                    "qwen_embedding": ("qwen3",)}\n        if encoder.config.model_type not in expected[kind]:\n            raise ValueError(f"Unexpected encoder architecture for {kind}")\n        self.encoder, self.kind = encoder, kind\n        if hasattr(encoder, "pooler"):\n            encoder.pooler = None\n        if hasattr(encoder.config, "use_cache"):\n            encoder.config.use_cache = False\n        self.dropout_probability = float(dropout)\n        self.dropout = nn.Dropout(self.dropout_probability)\n        self.classifier = nn.Linear(encoder.config.hidden_size, len(TARGET_SYMPTOMS))\n\n    def forward(self, input_ids, attention_mask, token_type_ids=None):\n        if (attention_mask.sum(1) == 0).any():\n            raise ValueError("Empty token sequence")\n        inputs = {"input_ids": input_ids, "attention_mask": attention_mask, "return_dict": True}\n        if self.kind in ("control", "klue_asl") and token_type_ids is not None:\n            inputs["token_type_ids"] = token_type_ids\n        if self.kind == "qwen_embedding":\n            positions = attention_mask.long().cumsum(-1) - 1\n            inputs.update(position_ids=positions.masked_fill(attention_mask == 0, 0), use_cache=False)\n        hidden = self.encoder(**inputs).last_hidden_state\n        if self.kind == "kure":\n            pooled = hidden[:, 0]\n        elif self.kind in ("control", "klue_asl"):\n            mask = attention_mask.unsqueeze(-1).to(hidden.dtype)\n            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1)\n        else:\n            positions = torch.arange(hidden.shape[1], device=hidden.device)\n            last = positions.unsqueeze(0).masked_fill(attention_mask == 0, -1).max(1).values\n            pooled = hidden[torch.arange(len(hidden), device=hidden.device), last]\n        return self.classifier(self.dropout(pooled))\n\n\ndef create_model(kind, spec, settings, device):\n    if kind not in KINDS or _get(spec, "kind", kind) != kind:\n        raise ValueError("Model kind and source specification disagree")\n    source = _get(spec, "snapshot_path") or _get(spec, "model_id")\n    if not source:\n        raise ValueError("Model source is missing")\n    local = Path(str(source)).is_dir()\n    revision = _get(spec, "revision")\n    if not local and not re.fullmatch(r"[0-9a-f]{40}", str(revision)):\n        raise ValueError("Remote model requires an immutable revision")\n    kwargs = {"trust_remote_code": False, "local_files_only": local}\n    if not local:\n        kwargs["revision"] = revision\n    tokenizer = AutoTokenizer.from_pretrained(str(source), use_fast=True, **kwargs)\n    if tokenizer.pad_token_id is None:\n        if kind != "qwen_embedding" or tokenizer.eos_token_id is None:\n            raise ValueError("Tokenizer needs a padding token")\n        tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = "right"\n    encoder_kwargs = dict(kwargs)\n    if kind != "qwen_embedding":\n        encoder_kwargs["add_pooling_layer"] = False\n    encoder = AutoModel.from_pretrained(str(source), **encoder_kwargs)\n    encoder.config.pad_token_id = tokenizer.pad_token_id\n    if kind in ("kure", "qwen_embedding"):\n        encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    return EncoderClassifier(encoder, kind, _get(settings, "dropout", .1)).to(device), tokenizer\n\n\ndef _tokenize(tokenizer, texts, maximum, kind="control"):\n    encoded = tokenizer(prepare_texts(kind, texts), truncation=True, max_length=int(maximum), padding=False)\n    keys = [key for key in ("input_ids", "attention_mask", "token_type_ids") if key in encoded]\n    return [{key: encoded[key][i] for key in keys} for i in range(len(texts))]\n\n\ndef _length_statistics(tokenizer, texts, rows, maximum, kind="control"):\n    lengths = []\n    for start in range(0, len(rows), 256):\n        encoded = tokenizer(prepare_texts(kind, [texts[int(i)] for i in rows[start:start + 256]]),\n                            truncation=False, padding=False, return_attention_mask=False,\n                            return_token_type_ids=False)\n        lengths.extend(len(value) for value in encoded["input_ids"])\n    lengths = np.asarray(lengths, dtype=np.int64)\n    return {"training_rows": len(rows), "max_length": maximum,\n            "raw_token_percentiles": {str(p): float(np.percentile(lengths, p)) for p in (0, 50, 90, 95, 99, 100)},\n            "truncated_training_rows": int((lengths > maximum).sum()),\n            "truncated_training_fraction": float(np.mean(lengths > maximum)),\n            "truncation": "right_truncation_after_fixed_input_format",\n            "preprocessing": model_protocol(kind)["preprocessing"]}\n\n\ndef _batch(tokenizer, tokens, rows, device):\n    packed = tokenizer.pad([tokens[int(i)] for i in rows], padding=True, return_tensors="pt")\n    return {key: value.to(device) for key, value in packed.items()\n            if key in ("input_ids", "attention_mask", "token_type_ids")}\n\n\ndef _optimizer(model, kind, settings):\n    groups = {}\n    for name, parameter in model.named_parameters():\n        if not parameter.requires_grad:\n            continue\n        lr = float(_get(settings, "head_lr", 1e-4) if name.startswith("classifier.")\n                   else _get(settings, f"{kind}_lr", 2e-5))\n        decay = 0. if name.endswith("bias") or "norm" in name.lower() else float(_get(settings, "weight_decay", .01))\n        if not math.isfinite(lr) or lr <= 0 or not math.isfinite(decay) or decay < 0:\n            raise ValueError("Invalid optimizer settings")\n        groups.setdefault((lr, decay), []).append(parameter)\n    return torch.optim.AdamW([{"params": params, "lr": lr, "weight_decay": decay}\n                              for (lr, decay), params in groups.items()])\n\n\ndef _rng():\n    numpy = np.random.get_state()\n    return {"python": random.getstate(), "numpy": (numpy[0], numpy[1].tolist(), numpy[2], numpy[3], numpy[4]),\n            "torch": torch.get_rng_state(),\n            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}\n\n\ndef _restore_rng(value):\n    random.setstate(value["python"])\n    numpy = value["numpy"]\n    np.random.set_state((numpy[0], np.asarray(numpy[1], dtype=np.uint32), numpy[2], numpy[3], numpy[4]))\n    torch.set_rng_state(value["torch"])\n    if value["cuda"]:\n        if len(value["cuda"]) != torch.cuda.device_count():\n            raise ValueError("Exact resume requires the same number of visible CUDA devices")\n        torch.cuda.set_rng_state_all(value["cuda"])\n\n\ndef _update(model, tokenizer, tokens, targets, rows, batch_size, optimizer, scaler, weights, device):\n    """One complete effective batch, replaying fp16 overflow without skipping rows."""\n    model.train()\n    rng = _rng()\n    for retry in range(9):\n        optimizer.zero_grad(set_to_none=True)\n        if retry:\n            _restore_rng(rng)\n        total_loss = 0.\n        for start in range(0, len(rows), batch_size):\n            part = rows[start:start + batch_size]\n            labels = torch.as_tensor(targets[part], dtype=torch.float32, device=device)\n            with _autocast(device):\n                logits = model(**_batch(tokenizer, tokens, part, device))\n                loss = classification_loss(logits, labels, weights, model.kind)\n            if not torch.isfinite(loss):\n                raise FloatingPointError("Training loss is not finite")\n            scaler.scale(loss * (len(part) / len(rows))).backward()\n            total_loss += float(loss.detach()) * len(part)\n        scaler.unscale_(optimizer)\n        norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n        if not scaler.is_enabled() and not torch.isfinite(norm):\n            raise FloatingPointError("Training gradients are not finite")\n        scale = scaler.get_scale()\n        scaler.step(optimizer)\n        scaler.update()\n        optimizer.zero_grad(set_to_none=True)\n        if scaler.get_scale() >= scale:\n            if not torch.isfinite(norm):\n                raise FloatingPointError("Accepted update has nonfinite gradients")\n            return total_loss / len(rows), retry\n    raise FloatingPointError("FP16 overflow retry limit exceeded; no rows were silently skipped")\n\n\ndef _autotune(kind, spec, settings, texts, targets, rows, device, budget):\n    effective = int(_get(settings, "effective_batch_size", 16))\n    candidates = sorted(set(int(value) for value in _get(settings, "batch_candidates", [1, 2, 4, 8, 16])))\n    if not candidates or any(value <= 0 or effective % value for value in candidates):\n        raise ValueError("Batch candidates must be positive divisors of effective batch size")\n    records = []\n    maximum = int(_get(settings, f"{kind}_max_length", _default_maximum(kind)))\n    for size in candidates:\n        if budget is not None:\n            budget.check("batch_autotune", training=True)\n        seed_everything(_get(settings, "seed", 42))\n        model = optimizer = scaler = None\n        try:\n            model, tokenizer = create_model(kind, spec, settings, device)\n            tokens = _tokenize(tokenizer, texts, maximum, kind)\n            optimizer = _optimizer(model, kind, settings)\n            scaler = torch.amp.GradScaler("cuda", enabled=_amp_dtype(device) == torch.float16)\n            weights = positive_weights(targets[rows]).to(device)\n            if device.type == "cuda":\n                torch.cuda.reset_peak_memory_stats(device)\n            # The first real step allocates Adam state and checks the longest rows.\n            longest = sorted(rows.tolist(), key=lambda i: len(tokens[i]["input_ids"]), reverse=True)\n            longest = np.resize(np.asarray(longest, dtype=np.int64), effective)\n            _update(model, tokenizer, tokens, targets, longest, size, optimizer, scaler, weights, device)\n            rng = np.random.default_rng(int(_get(settings, "seed", 42)))\n            probe = np.resize(rng.permutation(rows), max(effective * 2, int(_get(settings, "benchmark_examples", 64))))\n            measured = 0\n            _sync(device)\n            started = time.monotonic()\n            for start in range(0, len(probe), effective):\n                if budget is not None:\n                    budget.check("batch_autotune_measured_update", training=True)\n                part = probe[start:start + effective]\n                _update(model, tokenizer, tokens, targets, part, size, optimizer, scaler, weights, device)\n                measured += len(part)\n            _sync(device)\n            elapsed = time.monotonic() - started\n            records.append({"batch_size": size, "status": "ok", "examples": measured,\n                            "seconds": elapsed, "examples_per_second": measured / elapsed,\n                            "peak_vram_bytes": int(torch.cuda.max_memory_allocated(device)) if device.type == "cuda" else 0})\n        except torch.cuda.OutOfMemoryError:\n            records.append({"batch_size": size, "status": "out_of_memory"})\n        finally:\n            del model, optimizer, scaler\n            gc.collect()\n            if device.type == "cuda":\n                torch.cuda.empty_cache()\n    successful = [item for item in records if item["status"] == "ok"]\n    if not successful:\n        raise RuntimeError("No batch candidate fits this GPU at the requested sequence length")\n    best = max(successful, key=lambda item: item["examples_per_second"])\n    return {"selected_batch_size": best["batch_size"], "effective_batch_size": effective,\n            "gradient_accumulation_steps": effective // best["batch_size"],\n            "seconds_per_training_example": 1. / best["examples_per_second"], "candidates": records}\n\n\ndef _sha(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _json(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n    os.replace(temporary, path)\n\n\ndef _atomic_torch_save(value, path):\n    path = Path(path)\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save(value, temporary)\n    os.replace(temporary, path)\n\n\ndef _cpu_state(module):\n    return {name: value.detach().cpu() for name, value in module.state_dict().items()}\n\n\ndef _identity(kind, spec, settings, texts, targets, train_rows, dev_rows, epochs):\n    digest = hashlib.sha256()\n    for text in texts:\n        raw = text.encode("utf-8")\n        digest.update(len(raw).to_bytes(8, "little"))\n        digest.update(raw)\n    digest.update(targets.tobytes())\n    training_keys = {name: _get(settings, name, default) for name, default in {\n        "seed": 42, "effective_batch_size": 16, "head_lr": 1e-4, "weight_decay": .01,\n        "warmup_ratio": .05, "dropout": .1, f"{kind}_lr": 2e-5,\n        f"{kind}_max_length": _default_maximum(kind)}.items()}\n    return {"format": FORMAT, "kind": kind, "model_id": _get(spec, "model_id"),\n            "revision": _get(spec, "revision"), "epochs": epochs, "settings": training_keys,\n            "data_sha256": digest.hexdigest(), "train_indices": train_rows.tolist(),\n            "dev_indices": None if dev_rows is None else dev_rows.tolist(),\n            "target_symptoms": list(TARGET_SYMPTOMS), "decision": DECISION,\n            "model_protocol": model_protocol(kind)}\n\n\ndef _save_assets(path, model, tokenizer, metadata):\n    metadata = {**metadata, "model_protocol": model_protocol(model.kind)}\n    directory = Path(path)\n    directory.mkdir(parents=True, exist_ok=True)\n    model.encoder.config.save_pretrained(directory / "encoder_config")\n    tokenizer.save_pretrained(directory / "tokenizer")\n    _atomic_torch_save(_cpu_state(model), directory / "model.pt")\n    files = {p.relative_to(directory).as_posix(): _sha(p) for p in sorted(directory.rglob("*"))\n             if p.is_file() and p.name != "manifest.json"}\n    _json(directory / "manifest.json", {"format": FORMAT, "target_symptoms": list(TARGET_SYMPTOMS),\n                                        "decision": DECISION, "metadata": metadata, "files": files})\n\n\ndef _save_reload_reference(path, model, tokenizer, tokens, rows, batch_size, device):\n    rng = _rng()\n    try:\n        selected = np.asarray(rows[:16], dtype=np.int64)\n        logits = _predict(model, tokenizer, tokens, selected, batch_size, device)\n        temporary = Path(path).with_name(Path(path).name + ".tmp")\n        with temporary.open("wb") as handle:\n            np.savez(handle, indices=selected, logits=logits, batch_size=np.asarray(batch_size, dtype=np.int64))\n        os.replace(temporary, path)\n    finally:\n        _restore_rng(rng)\n\n\ndef load_checkpoint(checkpoint_dir, device="cpu"):\n    directory = Path(checkpoint_dir).resolve()\n    manifest = json.loads((directory / "manifest.json").read_text(encoding="utf-8"))\n    if (manifest.get("format") != FORMAT or manifest.get("target_symptoms") != list(TARGET_SYMPTOMS)\n            or manifest.get("decision") != DECISION):\n        raise ValueError("Checkpoint format, label order, or decision policy mismatch")\n    files = manifest["files"]\n    actual = {p.relative_to(directory).as_posix() for p in directory.rglob("*") if p.is_file() and p.name != "manifest.json"}\n    if actual != set(files):\n        raise ValueError("Unexpected or missing checkpoint files")\n    for relative, digest in files.items():\n        path = directory / relative\n        if path.is_symlink() or not path.resolve().is_relative_to(directory) or _sha(path) != digest:\n            raise ValueError("Checkpoint asset integrity failure")\n    metadata = manifest["metadata"]\n    if metadata.get("model_protocol") != model_protocol(metadata.get("kind")):\n        raise ValueError("Checkpoint objective or preprocessing protocol mismatch")\n    if metadata.get("pooling") != model_protocol(metadata["kind"])["pooling"]:\n        raise ValueError("Checkpoint pooling protocol mismatch")\n    config = AutoConfig.from_pretrained(directory / "encoder_config", local_files_only=True, trust_remote_code=False)\n    kwargs = {} if metadata["kind"] == "qwen_embedding" else {"add_pooling_layer": False}\n    model = EncoderClassifier(AutoModel.from_config(config, trust_remote_code=False, **kwargs),\n                              metadata["kind"], metadata["dropout"])\n    model.load_state_dict(torch.load(directory / "model.pt", map_location="cpu", weights_only=True), strict=True)\n    tokenizer = AutoTokenizer.from_pretrained(directory / "tokenizer", local_files_only=True,\n                                               trust_remote_code=False, use_fast=True)\n    return model.to(device), tokenizer, metadata\n\n\n@torch.no_grad()\ndef _predict(model, tokenizer, tokens, rows, batch_size, device, budget=None):\n    model.eval()\n    chunks = []\n    for start in range(0, len(rows), batch_size):\n        if budget is not None:\n            budget.check("classifier_evaluation", training=True)\n        with _autocast(device):\n            logits = model(**_batch(tokenizer, tokens, rows[start:start + batch_size], device))\n        values = logits.float().cpu().numpy()\n        if not np.isfinite(values).all():\n            raise FloatingPointError("Prediction logits are not finite")\n        chunks.append(values)\n    return np.concatenate(chunks).astype(np.float32)\n\n\ndef predict_checkpoint(checkpoint_dir, texts, indices=None, *, device="cpu", batch_size=8, budget=None):\n    if not isinstance(batch_size, int) or isinstance(batch_size, bool) or batch_size < 1:\n        raise ValueError("Prediction batch size must be a positive integer")\n    model, tokenizer, metadata = load_checkpoint(checkpoint_dir, device)\n    try:\n        rows = _indices(np.arange(len(texts)) if indices is None else indices, len(texts))\n        tokens = _tokenize(tokenizer, texts, metadata["max_length"], metadata["kind"])\n        return _predict(model, tokenizer, tokens, rows, batch_size, device, budget)\n    finally:\n        del model\n        gc.collect()\n        if torch.device(device).type == "cuda":\n            torch.cuda.empty_cache()\n\n\ndef _macro_f1(targets, logits):\n    predicted = logits >= 0\n    gold = targets.astype(bool)\n    tp = np.sum(predicted & gold, axis=0)\n    denom = np.sum(predicted, axis=0) + np.sum(gold, axis=0)\n    return float(np.divide(2 * tp, denom, out=np.zeros(9, dtype=float), where=denom != 0).mean())\n\n\ndef train_model(kind, spec, settings, texts, targets, train_indices, dev_indices,\n                output_dir, budget, epochs, resume=False):\n    """Train one candidate; pause only at optimizer boundaries, never lose rows.\n\n    ``epochs`` is the fixed scheduler horizon and must remain identical on resume.\n    A paused evaluation restarts that evaluation, not the completed Training epoch.\n    """\n    if kind not in KINDS or isinstance(epochs, bool) or not isinstance(epochs, int) or epochs < 1:\n        raise ValueError("Invalid model kind or epoch count")\n    targets = _targets(targets, len(texts))\n    rows = _indices(train_indices, len(texts))\n    dev = None if dev_indices is None else _indices(dev_indices, len(texts))\n    if dev is not None and np.intersect1d(rows, dev).size:\n        raise ValueError("Training and development rows must be disjoint")\n    directory = Path(output_dir)\n    directory.mkdir(parents=True, exist_ok=True)\n    result_path, state_path = directory / "training_result.json", directory / "training_state.pt"\n    identity = _identity(kind, spec, settings, texts, targets, rows, dev, epochs)\n    identity_path = directory / "training_identity.json"\n    if identity_path.exists():\n        if json.loads(identity_path.read_text(encoding="utf-8")) != identity:\n            raise ValueError("Resume identity differs in data, model, labels, settings, or scheduler horizon")\n        if not resume:\n            raise FileExistsError("Training output already exists; choose resume or a fresh output directory")\n    elif resume:\n        raise FileNotFoundError("Resume requires an existing training identity")\n    else:\n        _json(identity_path, identity)\n    if resume and result_path.exists():\n        result = json.loads(result_path.read_text(encoding="utf-8"))\n        if result["status"] == "complete":\n            result["checkpoint_dir"] = str(directory / "best_model")\n            if not (directory / "best_model" / "manifest.json").is_file():\n                raise FileNotFoundError("Completed training result has no portable model checkpoint")\n            _json(result_path, result)\n            return result\n    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")\n    if device.type != "cuda" and not bool(_get(settings, "allow_cpu", False)):\n        raise RuntimeError("Training requires a GPU; CPU is only enabled for explicit local tests")\n    seed = int(_get(settings, "seed", 42))\n    effective = int(_get(settings, "effective_batch_size", 16))\n    if effective < 1:\n        raise ValueError("Effective batch size must be positive")\n    maximum = int(_get(settings, f"{kind}_max_length", _default_maximum(kind)))\n    tuning_path = directory / "batch_benchmark.json"\n    started = time.monotonic()\n    try:\n        if tuning_path.exists():\n            tuning = json.loads(tuning_path.read_text(encoding="utf-8"))\n        else:\n            tuning = _autotune(kind, spec, settings, texts, targets, rows, device, budget)\n            _json(tuning_path, tuning)\n    except BudgetExceeded:\n        result = {"status": "paused", "best_epoch": None, "best_score": None,\n                  "checkpoint_dir": str(directory / "best_model"), "runtime": {"stage": "batch_autotune"}}\n        _json(result_path, result)\n        return result\n    batch_size = int(tuning["selected_batch_size"])\n    seed_everything(seed)  # Benchmark updates and RNG never become Training initialization.\n    model, tokenizer = create_model(kind, spec, settings, device)\n    tokens = _tokenize(tokenizer, texts, maximum, kind)\n    if not (directory / "length_statistics.json").exists():\n        _json(directory / "length_statistics.json", _length_statistics(tokenizer, texts, rows, maximum, kind))\n    optimizer = _optimizer(model, kind, settings)\n    total_updates = math.ceil(len(rows) / effective) * epochs\n    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_updates * float(_get(settings, "warmup_ratio", .05))), total_updates)\n    scaler = torch.amp.GradScaler("cuda", enabled=_amp_dtype(device) == torch.float16)\n    weights = positive_weights(targets[rows]).to(device)\n    progress = {"epoch": 0, "next_update": 0, "phase": "train", "global_updates": 0,\n                "best_epoch": None, "best_score": None, "history": [], "epoch_loss_sum": 0.,\n                "epoch_examples": 0, "overflow_retries": 0, "training_seconds": 0.}\n    if resume and state_path.exists():\n        state = torch.load(state_path, map_location="cpu", weights_only=True)\n        if state["identity"] != identity or state["batch_size"] != batch_size:\n            raise ValueError("Training checkpoint identity or batch size mismatch")\n        if state["precision"] != str(_amp_dtype(device) or torch.float32):\n            raise ValueError("Exact resume requires the same mixed-precision mode")\n        model.load_state_dict(state["model"], strict=True)\n        optimizer.load_state_dict(state["optimizer"])\n        scheduler.load_state_dict(state["scheduler"])\n        scaler.load_state_dict(state["scaler"])\n        progress = state["progress"]\n        _restore_rng(state["rng"])\n        del state\n    remaining_rows = (epochs - progress["epoch"]) * len(rows) - progress["epoch_examples"]\n    estimated_minutes = float(tuning["seconds_per_training_example"]) * remaining_rows / 60.\n    print(f"{kind}: selected batch {batch_size}, effective batch {effective}; measured estimate: "\n          f"{estimated_minutes:.1f} remaining training minutes (excludes download, tokenization, evaluation and checkpoint saves).", flush=True)\n\n    def save_state():\n        _atomic_torch_save({"identity": identity, "batch_size": batch_size,\n                           "precision": str(_amp_dtype(device) or torch.float32),\n                           "model": _cpu_state(model), "optimizer": optimizer.state_dict(),\n                           "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),\n                           "progress": progress, "rng": _rng()}, state_path)\n\n    status = "complete"\n    try:\n        while progress["epoch"] < epochs:\n            epoch = int(progress["epoch"])\n            order = np.random.default_rng(seed + epoch).permutation(rows)\n            if progress["phase"] == "train":\n                for update in range(progress["next_update"], math.ceil(len(rows) / effective)):\n                    if budget is not None:\n                        budget.check("classifier_training", training=True)\n                    group = order[update * effective:(update + 1) * effective]\n                    _sync(device)\n                    update_started = time.monotonic()\n                    loss, retries = _update(model, tokenizer, tokens, targets, group, batch_size,\n                                             optimizer, scaler, weights, device)\n                    scheduler.step()\n                    _sync(device)\n                    progress["training_seconds"] += time.monotonic() - update_started\n                    progress["next_update"] = update + 1\n                    progress["global_updates"] += 1\n                    progress["epoch_loss_sum"] += loss * len(group)\n                    progress["epoch_examples"] += len(group)\n                    progress["overflow_retries"] += retries\n                    every = int(_get(settings, "checkpoint_every_updates", 100))\n                    if every > 0 and progress["global_updates"] % every == 0:\n                        save_state()\n                    log_every = int(_get(settings, "log_every", 50))\n                    if log_every > 0 and progress["global_updates"] % log_every == 0:\n                        print(f"{kind}: epoch {epoch + 1}/{epochs}, update {progress[\'global_updates\']}/{total_updates}, loss {loss:.5f}", flush=True)\n                progress["phase"] = "evaluate"\n                save_state()\n            score = None\n            if dev is not None:\n                logits = _predict(model, tokenizer, tokens, dev, batch_size, device, budget)\n                score = _macro_f1(targets[dev], logits)\n                np.save(directory / f"epoch_{epoch + 1}_development_logits.npy", logits)\n            improves = dev is None or progress["best_score"] is None or score > progress["best_score"]\n            if improves:\n                progress["best_epoch"], progress["best_score"] = epoch + 1, score\n                metadata = {"kind": kind, "model_id": _get(spec, "model_id"), "revision": _get(spec, "revision"),\n                            "max_length": maximum, "dropout": model.dropout_probability,\n                            "selected_epochs": epoch + 1, "training_scope": "all_training" if dev is None else "fit_only",\n                            "pos_weight": weights.cpu().tolist(), "decision": DECISION,\n                            "pooling": model_protocol(kind)["pooling"]}\n                _save_assets(directory / "best_model", model, tokenizer, metadata)\n                _save_reload_reference(directory / "reload_reference.npz", model, tokenizer, tokens,\n                                       rows if dev is None else dev, batch_size, device)\n            progress["history"].append({"epoch": epoch + 1, "macro_f1": score,\n                                         "train_mean_loss": progress["epoch_loss_sum"] / progress["epoch_examples"],\n                                         "train_examples": progress["epoch_examples"]})\n            progress.update(epoch=epoch + 1, next_update=0, phase="train", epoch_loss_sum=0., epoch_examples=0)\n            save_state()\n    except BudgetExceeded:\n        status = "paused"\n        save_state()\n    finally:\n        optimizer.zero_grad(set_to_none=True)\n    result = {"status": status, "best_epoch": progress["best_epoch"], "best_score": progress["best_score"],\n              "checkpoint_dir": str(directory / "best_model"), "history": progress["history"],\n              "runtime": {"invocation_seconds": time.monotonic() - started,\n                          "training_seconds": progress["training_seconds"], "global_updates": progress["global_updates"],\n                          "overflow_retries": progress["overflow_retries"], "batch_size": batch_size,\n                          "effective_batch_size": effective, "precision": str(_amp_dtype(device) or torch.float32),\n                          "device": str(device), "epoch": progress["epoch"], "phase": progress["phase"]}}\n    _json(directory / "history.json", progress["history"])\n    _json(result_path, result)\n    if status == "complete" and state_path.exists():\n        state_path.unlink()  # Only this trainer\'s now-obsolete resumable optimizer state.\n    del model, optimizer, scheduler, scaler\n    gc.collect()\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n    return result\n', 'model_specs.json': '{\n  "kure": {\n    "model_id": "nlpai-lab/KURE-v1",\n    "revision": "main",\n    "revision_policy": "Resolve once to an immutable commit before training, then preserve for resume and refit.",\n    "license": "MIT",\n    "source": "https://huggingface.co/nlpai-lab/KURE-v1",\n    "model_card": "https://huggingface.co/nlpai-lab/KURE-v1",\n    "license_note": "KURE-v1 derives from BAAI/bge-m3. Preserve upstream model cards and license notices."\n  },\n  "control": {\n    "model_id": "klue/roberta-base",\n    "revision": "02f94ba5e3fcb7e2a58a390b8639b0fac974a8da",\n    "license": "CC-BY-SA-4.0",\n    "source": "https://huggingface.co/klue/roberta-base",\n    "model_card": "https://huggingface.co/klue/roberta-base",\n    "license_note": "Preserve attribution and share-alike terms when redistributing fine-tuned weights."\n  },\n  "klue_asl": {\n    "model_id": "klue/roberta-base",\n    "revision": "02f94ba5e3fcb7e2a58a390b8639b0fac974a8da",\n    "license": "CC-BY-SA-4.0",\n    "source": "https://huggingface.co/klue/roberta-base",\n    "model_card": "https://huggingface.co/klue/roberta-base",\n    "license_note": "Same public encoder as control; a new full-encoder weighted asymmetric-loss experiment. Preserve attribution and share-alike terms."\n  },\n  "qwen_embedding": {\n    "model_id": "Qwen/Qwen3-Embedding-0.6B",\n    "revision": "main",\n    "revision_policy": "Resolve once to an immutable commit before training, then preserve for resume and refit.",\n    "license": "Apache-2.0",\n    "source": "https://huggingface.co/Qwen/Qwen3-Embedding-0.6B",\n    "model_card": "https://huggingface.co/Qwen/Qwen3-Embedding-0.6B",\n    "license_note": "Preserve upstream license and model card. Fixed English task instruction and last-token pooling are used; retrieval scores are not symptom F1."\n  }\n}\n', 'requirements.txt': "# Keep Kaggle's compatible CUDA-enabled PyTorch and numerical packages.\ntorch>=2.6,<3\ntransformers==4.57.1\nnumpy>=1.26,<3\nscikit-learn>=1.6,<2\nscipy>=1.11,<2\niterative-stratification==0.1.9\njoblib>=1.3,<2\nsentencepiece>=0.2,<0.3\nprotobuf>=4.25,<7\naccelerate==1.10.1\n", 'README.md': '# Mission 3 ver29\n\nKaggle에서 새 KLUE-ASL과 Qwen3-Embedding 분류기를 학습하고, 완료된 ver28의 저장 예측을 기준으로 비교합니다. 처음 사용하는 경우 [Kaggle 실행 안내](../../../docs/mission3_ver29_kaggle_guide.md)를 참고합니다. 목표는 Macro F1 0.7 초과지만, 실제 GPU 학습과 목표 점수 달성은 실행 결과로 확인해야 합니다.\n\n## 실행 파일\n\n- `../mission3_ver29.ipynb`: 모든 실행 코드를 포함한 독립 노트북입니다.\n- 프로젝트 루트의 `outputs/mission3_ver29/mission3_ver29_inputs.zip`: 원본 Training 29,200개 / Validation 3,640개 JSON과 완료된 ver28 분석 ZIP을 모은 입력입니다. 각 파일의 원본 바이트를 보존하며 이전 모델 가중치는 필요하지 않습니다.\n- `requirements.txt`: ver27에서 사용하던 패키지의 부분집합입니다. 호환되는 Kaggle CUDA PyTorch와 수치 패키지는 유지하고 실제 버전을 기록합니다. PEFT / bitsandbytes는 필요하지 않습니다.\n\nKaggle의 Private Dataset에 ZIP을 올린 뒤 노트북에 Add Input으로 연결합니다. ZIP 또는 Kaggle이 압축을 푼 디렉터리를 자동으로 찾습니다. GPU T4 ×2와 Internet On을 설정하고 **Save Version → Save & Run All**을 한 번 실행합니다. 이 구현은 GPU 한 개를 사용하며 두 모델을 순서대로 학습합니다.\n\n## 두 실행 단계\n\n`RUN_MODE="screening"`이 기본값입니다. seed42 / 중복 그룹을 유지한 기존 70/15/15 분할에서 fit 20,440개로 KLUE-ASL과 Qwen3-Embedding을 최대 3 epoch 학습합니다. development 4,380개에서 최고 epoch와 아래 사전 고정 구성을 비교하고 선택을 고정합니다. confirmation 4,380개는 고정한 선택의 추가 확인에 사용하며 재선택에 사용하지 않습니다. 두 검증 분할 모두 과거 실험의 사용 이력이 있습니다. 공식 Validation은 이 단계에서 평가하지 않습니다.\n\n| 구성 | 확률 평균 |\n|---|---|\n| 기존 ver28 | 이전 KLUE + KURE, 1:1 |\n| KLUE-ASL | 새 KLUE 단독 |\n| Qwen3-Embedding | 새 Qwen 단독 |\n| 새 모델 평균 | KLUE-ASL + Qwen, 1:1 |\n| 4개 모델 평균 | 이전 KLUE + KURE + KLUE-ASL + Qwen, 1:1:1:1 |\n\nKLUE-ASL은 기존 masked mean pooling과 선형 9개 출력 구조를 유지하며 encoder 전체를 미세조정합니다. loss의 음성 항에만 sigmoid 확률의 제곱을 곱해 쉬운 음성 사례의 비중을 낮춥니다. 양성 gamma 0, 음성 gamma 2, clipping 0, 기존 sqrt 양성 가중치를 유지하는 **weighted ASL 변형**입니다. focal factor를 detach하지 않아 해당 항에도 역전파하며 논문 공식 구현을 그대로 재현한 모델은 아닙니다. 라벨별 attention은 사용하지 않습니다.\n\nver28의 T4 BF16 실행과 새 후보의 FP16 실행은 정밀도도 다르므로, 점수 차이를 ASL loss만의 효과로 해석할 수 없습니다.\n\nQwen3-Embedding은 고정 English instruction prefix, 마지막 유효 token pooling과 9개 선형 출력을 사용하며 encoder 전체를 미세조정합니다. loss는 기존 weighted BCE입니다. 임베딩 검색용 모델의 분류 미세조정 실험으로 성능 개선을 전제하지 않습니다.\n\n`REFERENCE_RUN=""`이면 패키지의 완료된 ver28 분석 ZIP을 자동 탐색합니다. 원본 데이터·분할·정답·참조 예측의 호환성을 검증합니다. 첫 screening에는 이 작은 분석 ZIP만 필요하지만 ver29 재개와 full에는 전체 ver29 Output이 필요합니다.\n\n결과를 검토한 후 **새 실행**에서 `RUN_MODE="full"`로 바꾸고 `SOURCE_RUN`에 완료된 screening 실행 폴더를 지정합니다. 선택한 구성과 epoch로 1개, 2개 또는 4개 모델을 전체 Training 29,200개에 재학습한 뒤 공식 Validation 평가, 저장 모델 재로딩 확인, 제출용 파일 생성을 진행합니다. full은 새로 초기화하며 학습률 일정도 선택된 epoch 길이로 다시 구성하므로 screening의 중간 checkpoint와 동일한 학습 궤적은 아닙니다. 이전 ver28 모델이 선택된 구성에 포함되면 해당 모델도 다시 학습합니다. 새 노트북에 ver29 입력 Dataset과 완료된 ver29 screening Output 하나를 모두 Add Input으로 연결해야 합니다. 실행 폴더가 하나면 `SOURCE_RUN="/kaggle/input"`을 사용할 수 있고, 여러 개면 `run_state.json`이 있는 정확한 폴더를 지정합니다. 전체 학습은 screening보다 오래 걸릴 수 있습니다.\n\n외부 학습 데이터, 정답 수정, 표본 임의 삭제, threshold / offset / 혼합비 검색은 하지 않습니다. 입력은 통화 전사 텍스트이며 원본 9개 증상 정답을 사용합니다. 양성 가중치는 fit에서만 계산하며 모든 후보의 판정 기준은 0.5입니다. 전체 Training을 이미 학습한 ver25 / 친구 모델은 내부 검증 앙상블에 포함하지 않습니다.\n\n## 시간과 재개\n\n최대 480분은 설치·모델 다운로드·준비·평가·저장을 포함합니다. 배치는 실제 처리량으로 정합니다. T4에서는 native FP16을 사용하고 native BF16 지원 GPU에서만 BF16을 선택합니다. ver28의 BF16 에뮬레이션 선택 문제를 수정했지만 실제 속도 향상은 GPU 실행에서 확인해야 합니다. 일정한 scheduler 길이를 유지하고 남은 시간이 부족하면 checkpoint를 저장해 `paused`로 종료합니다. 시간 때문에 epoch 수를 줄인 뒤 완료로 표시하지 않습니다.\n\n이전 저장 버전의 Output을 새 노트북에 연결하고 `RESUME_RUN`에 중단된 실행 폴더를 넣으면 같은 모드로 이어갑니다. 재개에는 가중치뿐 아니라 optimizer / scheduler / 혼합 정밀도 / 난수 및 진행 상태가 필요합니다. 따라서 분석 ZIP만으로 재개할 수 없습니다. 이전 입력은 읽기 전용으로 유지하고 새 실행 폴더에 결과를 저장합니다.\n\n이전 ver29 Output 하나만 연결하면 `RESUME_RUN="/kaggle/input"`으로 자동 탐색할 수 있습니다. 여러 실행 폴더가 있으면 구체적인 경로가 필요합니다. ver28 checkpoint를 ver29의 재개 입력으로 사용하지 마세요.\n\n데이터 추출물·Hugging Face 캐시는 `/kaggle/temp/`에 두고 결과는 `/kaggle/working/results/mission3/ver29/실행시각_고유번호/`에 저장합니다. 임시 캐시는 다음 실행에서 다시 다운로드될 수 있습니다. Kaggle 강제 종료 전에 기록하지 못한 진행은 보존을 보장할 수 없습니다.\n\n## 상태와 보관\n\n| 상태 | 뜻 |\n|---|---|\n| `screening_complete` | 후보 비교·선택·추가 확인 완료. 최종 제출 모델은 아님 |\n| `paused` | 저장 후 시간 제한으로 중단. 같은 모드에서 재개 필요 |\n| `complete` + `submission_ready=true` | 전체 학습과 제출 준비 완료 |\n| `failed` | 오류. 완료 결과로 사용하지 않음 |\n| `interrupted` | 강제 종료. checkpoint와 manifest가 완전히 저장됐는지 확인 필요 |\n\n`run_state.json`을 먼저 확인합니다. 지표·예측·선택 근거·재현 코드와 모델·checkpoint를 실행 폴더에 보관합니다. 노트북 마지막에는 `ver29_analysis_실행번호.zip`도 만듭니다. 이 ZIP은 분석을 위한 작은 자료 모음이며 **큰 가중치·재개 checkpoint를 포함하지 않습니다**. Kaggle Output의 전체 실행 폴더를 지우지 마세요.\n\n## 재생성과 검증\n\n프로젝트 루트에서 노트북을 재생성합니다.\n\n```bash\npython src/mission3/mission3_ver29/build_notebook.py\n```\n\n빌더는 현재 ver29의 모든 실행 모듈과 설정·README·requirements를 포함하고 Python 소스와 노트북 코드 셀의 구문을 검사합니다. 테스트 파일과 결과 파일은 포함하지 않습니다. GPU·인터넷·Kaggle 계정·실제 F1은 이 로컬 검사로 검증되지 않습니다. 런타임 코드가 바뀌면 노트북도 다시 생성해야 합니다.\n\n필요한 패키지가 있는 환경에서 로컬 테스트를 실행합니다.\n\n```bash\npython -m unittest src.mission3.mission3_ver29.test_runtime src.mission3.mission3_ver29.test_trainer src.mission3.mission3_ver29.test_package_inputs src.mission3.mission3_ver29.test_reference -v\n```\n\nfull 완료 결과에는 `INFERENCE_README.md`, `inference.py`, `submission_manifest.json`, `source_snapshot/`과 모델 폴더가 함께 포함됩니다. 전체 실행 폴더에서 대회 인자 형식으로 실행합니다.\n\n```bash\npython inference.py --audio_dir /path/to/wavs --label_dir /path/to/jsons --ckpt_path ./submission_manifest.json --output ./outputs/mission3.csv\n```\n\n`--ckpt_path`는 `submission_manifest.json` 파일 또는 완료된 실행 폴더를 받습니다. 참조하는 모델·토크나이저·`source_snapshot/`도 같은 전체 폴더 구조로 보관해야 합니다. `--audio_dir`는 대회 인터페이스 호환을 위해 받지만 읽지 않습니다. 추론은 `--label_dir`의 전사 텍스트만 사용하며 `symptom` 정답을 읽지 않습니다. CUDA가 사용 가능하면 자동 선택하고 그렇지 않으면 CPU를 사용합니다. `--device cpu` 또는 `--device cuda`로 명시할 수도 있습니다.\n\n기존 module 실행과 dashed 인자도 지원합니다.\n\n```bash\npython -m source_snapshot.inference --run-dir . --label-dir /path/to/label_jsons --output predictions.csv --device cuda --batch-size 1\n```\n\n모델과 토크나이저는 로컬 파일에서 불러옵니다. 분석 ZIP만으로는 추론할 수 없습니다.\n\n모델 출처와 revision / 라이선스는 `model_specs.json` 및 각 실행의 snapshot에 기록합니다.\n'}
SUPPORT_SOURCE = '"""Persist launcher records outside immutable ver29 run manifests."""\nfrom __future__ import annotations\n\nimport codecs\nfrom datetime import datetime, timezone\nimport hashlib\nimport importlib.util\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport uuid\nimport zipfile\n\nINPUT_SHA256 = "cfa2f2e0a63a5788848e175f92b685ac7b36ca62b99142664a3b589d2986f7ff"\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as reader:\n        for block in iter(lambda: reader.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")\n    try:\n        with temporary.open("w", encoding="utf-8") as writer:\n            json.dump(value, writer, ensure_ascii=False, indent=2, allow_nan=False)\n            writer.flush()\n            os.fsync(writer.fileno())\n        temporary.replace(path)\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef volume_path(volume_dir):\n    root = Path(volume_dir).expanduser()\n    if not root.is_absolute() or not root.is_dir():\n        raise FileNotFoundError(\n            "Volume 폴더가 없습니다. Files에서 기존 Volume을 연결하고 VOLUME_DIR를 확인하세요.")\n    # Never create a missing mount as an ephemeral directory.\n    return root.resolve()\n\n\ndef writable_dir(root, relative):\n    path = root / relative\n    if not path.resolve().is_relative_to(root):\n        raise ValueError("저장 경로가 Volume 밖을 가리킵니다.")\n    current = path\n    while current != root:\n        if current.is_symlink():\n            raise ValueError("결과 저장 경로에 심볼릭 링크를 사용할 수 없습니다.")\n        current = current.parent\n    path.mkdir(parents=True, exist_ok=True)\n    return path\n\n\ndef input_path(root, explicit="", expected_sha=INPUT_SHA256):\n    path = Path(explicit).expanduser() if explicit else root / "mission3_ver29_inputs.zip"\n    if not path.is_absolute():\n        path = root / path\n    path = path.resolve()\n    if not path.is_relative_to(root) or not path.is_file():\n        raise FileNotFoundError(\n            "입력 ZIP이 없습니다. Volume 안에 mission3_ver29_inputs.zip을 업로드하세요. "\n            "하위 폴더에 올렸다면 INPUT_ZIP에 정확한 경로를 넣으세요.")\n    if file_hash(path) != expected_sha:\n        raise ValueError("입력 ZIP 검증 실패. 제공된 mission3_ver29_inputs.zip을 다시 업로드하세요.")\n    return path\n\n\ndef check_run_path(root, value, mode, resume=False):\n    if not value:\n        return ""\n    path = Path(value).expanduser()\n    if not path.is_absolute():\n        raise ValueError("SOURCE_RUN과 RESUME_RUN은 실행 폴더의 전체 경로여야 합니다.")\n    path = path.resolve()\n    if not path.is_relative_to(root) or not (path / "run_state.json").is_file():\n        raise ValueError("같은 Volume 안에서 run_state.json이 있는 정확한 실행 폴더를 지정하세요.")\n    state = json.loads((path / "run_state.json").read_text(encoding="utf-8"))\n    expected_status = "paused" if resume else "screening_complete"\n    expected_mode = mode if resume else "screening"\n    if state.get("status") != expected_status or state.get("run_mode") != expected_mode:\n        raise ValueError(f"선택한 실행은 {expected_mode}/{expected_status} 상태여야 합니다.")\n    return str(path)\n\n\ndef prepare(settings, volume_dir, scratch, *, expected_sha=INPUT_SHA256):\n    """Preflight before dependency installation; recipe validation stays in core."""\n    from mission3_ver29.settings import validate_settings\n\n    root = volume_path(volume_dir)\n    archive = input_path(root, settings.get("input_zip", ""), expected_sha)\n    mode = settings["run_mode"]\n    source = check_run_path(root, settings.get("source_run", ""), mode)\n    resume = check_run_path(root, settings.get("resume_run", ""), mode, resume=True)\n    if source and resume:\n        raise ValueError("SOURCE_RUN과 RESUME_RUN을 동시에 지정하지 마세요.")\n    settings = dict(settings)\n    settings.update(input_root=str(root), input_zip=str(archive), reference_run="",\n                    source_run=source, resume_run=resume,\n                    output_dir=str(root / "results/mission3/ver29"),\n                    cache_dir=str(root / "cache/huggingface"),\n                    work_dir=str(Path(scratch) / "data"), run_validation=mode == "full")\n    settings = validate_settings(settings)\n    invocation_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ_") + uuid.uuid4().hex[:8]\n    for relative in ("exports", "results/mission3/ver29", "cache/huggingface", "invocations"):\n        writable_dir(root, relative)\n    invocation = root / "invocations" / invocation_id\n    invocation.mkdir(exist_ok=False)\n    write_json(invocation / "settings.json", settings)\n    write_json(invocation / "launcher.json", {\n        "platform": "modal-notebook", "invocation_id": invocation_id,\n        "volume_dir": str(root), "input_sha256": expected_sha,\n        "status": "prepared", "submission_ready": False,\n    })\n    return settings, invocation\n\n\ndef install_command(requirements, constraints, torch_version, executable=None):\n    """Use Modal\'s uv when available; keep the installed CUDA torch unchanged."""\n    match = re.match(r"^(\\d+)\\.(\\d+)", torch_version)\n    if not match or not (int(match[1]) == 2 and int(match[2]) >= 6):\n        raise RuntimeError("CUDA PyTorch 2.6 이상 3 미만이 필요합니다. PyTorch를 자동 교체하지 않습니다.")\n    Path(constraints).write_text("torch==" + torch_version + "\\n", encoding="utf-8")\n    python = executable or sys.executable\n    uv = shutil.which("uv")\n    if uv:\n        command = [uv, "pip", "install", "--python", python]\n    elif importlib.util.find_spec("pip") is not None:\n        command = [python, "-m", "pip", "install", "--upgrade-strategy", "only-if-needed"]\n    else:\n        raise RuntimeError("uv와 pip를 찾지 못했습니다. Modal 기본 Python 이미지를 사용하세요.")\n    return command + ["--quiet", "-c", str(constraints), "-r", str(requirements)]\n\n\ndef run_logged(command, log_path, deadline, *, cwd=None, env=None, margin=10):\n    """Wait in the foreground, mirror UTF-8 output, and clean up the process group."""\n    from mission3_ver29.budget import BudgetExceeded, _stop_process\n\n    if deadline - time.monotonic() <= margin:\n        raise BudgetExceeded("subprocess startup", deadline_monotonic=deadline)\n    if not isinstance(command, (list, tuple)) or not command:\n        raise TypeError("command must be a nonempty argument sequence")\n    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")\n    # Refuse to truncate logs or silently start a duplicate cell invocation.\n    with Path(log_path).open("xb", buffering=0) as writer, Path(log_path).open("rb") as reader:\n        process = subprocess.Popen(command, cwd=cwd, env=env, stdout=writer,\n                                   stderr=subprocess.STDOUT, start_new_session=os.name == "posix")\n\n        def drain(final=False):\n            chunk = reader.read(256 * 1024)\n            while chunk:\n                print(decoder.decode(chunk), end="", flush=True)\n                chunk = reader.read(256 * 1024)\n            if final:\n                print(decoder.decode(b"", final=True), end="", flush=True)\n\n        try:\n            while process.poll() is None:\n                drain()\n                if time.monotonic() >= deadline - margin:\n                    raise BudgetExceeded("subprocess execution", deadline_monotonic=deadline)\n                time.sleep(0.2)\n            if process.returncode:\n                raise subprocess.CalledProcessError(process.returncode, command)\n        except BaseException:\n            _stop_process(process, min(deadline, time.monotonic() + margin))\n            raise\n        finally:\n            drain(final=True)\n            os.fsync(writer.fileno())\n    return 0\n\n\ndef collect_result(result_file, output_root, previous_states, failure=None):\n    """Attribute only this invocation\'s output; never bless a killed checkpoint."""\n    try:\n        result = json.loads(Path(result_file).read_text(encoding="utf-8"))\n        if not isinstance(result, dict):\n            raise ValueError("Invalid result")\n    except (OSError, ValueError):\n        result = {}\n    output_root = Path(output_root).resolve()\n    new_states = set(output_root.glob("*/run_state.json")) - set(previous_states)\n    new_dirs = {path.parent.resolve() for path in new_states}\n    run_dir = Path(result["run_dir"]).resolve() if result.get("run_dir") else None\n    if run_dir not in new_dirs:\n        result = {"status": "interrupted", "submission_ready": False}\n        if len(new_dirs) == 1:\n            result["run_dir"] = str(next(iter(new_dirs)))\n    else:\n        known = {"paused", "screening_complete", "complete", "failed"}\n        if result.get("status") not in known:\n            result.update(status="interrupted", submission_ready=False)\n        if result.get("status") != "complete":\n            result["submission_ready"] = False\n        # A successful runtime result already written before a launcher failure\n        # remains evidence, but the launcher failure is always reported separately.\n    if failure:\n        result["launcher_error_type"] = failure\n    write_json(result_file, result)\n    return result\n\n\ndef export_analysis(run_dir, exports):\n    """Export bounded metadata and predictions without changing the run manifest."""\n    run_dir = Path(run_dir).resolve()\n    exports = Path(exports)\n    exports.mkdir(parents=True, exist_ok=True)\n    archive = exports / ("ver29_analysis_" + run_dir.name + ".zip")\n    temporary = archive.with_suffix(".zip.tmp")\n    allowed = {".json", ".csv", ".npz", ".npy", ".md", ".txt", ".py"}\n    excluded = {"checkpoints", "checkpoint", "resume_checkpoint", "best_model", "model",\n                "tokenizer", "cache", "data"}\n    try:\n        with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as handle:\n            for path in sorted(run_dir.rglob("*")):\n                relative = path.relative_to(run_dir)\n                if (not path.is_file() or path.is_symlink() or path.suffix.lower() not in allowed\n                        or any(part in excluded for part in relative.parts)\n                        or not path.resolve().is_relative_to(run_dir)\n                        or path.stat().st_size > 20 * 1024 * 1024):\n                    continue\n                handle.write(path, (Path(run_dir.name) / relative).as_posix())\n        temporary.replace(archive)\n    finally:\n        temporary.unlink(missing_ok=True)\n    return archive\n\n\ndef execute(settings, invocation, package_parent, env, deadline):\n    """Finish launcher records/analysis even when the supervised child fails."""\n    invocation = Path(invocation)\n    output_root = Path(settings["output_dir"])\n    previous = set(output_root.glob("*/run_state.json"))\n    result_file = invocation / "result.json"\n    write_json(result_file, {"status": "starting", "submission_ready": False})\n    failure = None\n    try:\n        run_logged([sys.executable, "-u", "-m", "mission3_ver29.run_notebook",\n                    "--settings", str(invocation / "settings.json"), "--result-file", str(result_file)],\n                   invocation / "training.log", deadline, cwd=str(package_parent), env=env)\n    except BaseException as error:\n        failure = error\n    result = collect_result(result_file, output_root, previous,\n                            type(failure).__name__ if failure else None)\n    if result.get("run_dir"):\n        archive = export_analysis(result["run_dir"], invocation.parent.parent / "exports")\n        result["analysis_zip"] = str(archive)\n    write_json(invocation / "launcher.json", result)\n    print("\\n실행 상태:", result.get("status"))\n    print("결과 폴더:", result.get("run_dir", "생성되지 않음"))\n    print("분석 ZIP:", result.get("analysis_zip", "생성되지 않음"))\n    print("로그 폴더:", invocation)\n    if result.get("status") == "screening_complete":\n        print("비교 완료. 분석 ZIP을 확인한 뒤 full을 별도 실행하세요. 아직 제출 모델이 아닙니다.")\n    elif result.get("status") == "paused":\n        print("중간 저장 완료. 같은 L4와 RUN_MODE를 유지하고 위 결과 폴더를 RESUME_RUN에 넣으세요.")\n    elif result.get("status") == "complete" and result.get("submission_ready"):\n        print("전체 학습 및 제출 준비 완료. 공식 Validation 지표와 제출 manifest를 확인하세요.")\n    else:\n        print("완료 상태가 아닙니다. 오류 기록을 확인한 뒤 재개 여부를 판단하세요.")\n    print("분석 ZIP에는 모델 가중치가 없습니다. Volume의 전체 결과 폴더를 보관하세요.")\n    print("저장된 파일을 확인하고 Stop kernel로 GPU를 종료하세요.")\n    if failure:\n        if isinstance(failure, KeyboardInterrupt):\n            raise failure\n        raise RuntimeError("ver29 실행 오류: " + type(failure).__name__ + ". 위 로그 폴더를 확인하세요.") from None\n    if result.get("status") not in {"screening_complete", "paused", "complete"}:\n        raise RuntimeError("ver29 결과가 완성되지 않았습니다. result.json을 확인하세요.")\n    return result\n'
if "DEADLINE_MONOTONIC" not in globals():
    raise RuntimeError("첫 설정 셀부터 실행하세요.")
if time.monotonic() >= DEADLINE_MONOTONIC:
    raise RuntimeError("이번 실행 시간이 끝났습니다. 첫 설정 셀부터 다시 시작하세요.")
if os.name != "posix" or not Path("/mnt").is_dir():
    raise RuntimeError("Modal 노트북에서 L4와 Volume을 연결한 뒤 실행하세요.")
package_parent = Path(tempfile.mkdtemp(prefix="mission3_ver29_"))
package_dir = package_parent / "mission3_ver29"
package_dir.mkdir()
for name, source in RUNTIME_SOURCES.items():
    (package_dir / name).write_text(source, encoding="utf-8")
support_file = package_parent / "modal_support.py"
support_file.write_text(SUPPORT_SOURCE, encoding="utf-8")
for name in list(sys.modules):
    if name == "mission3_ver29" or name.startswith("mission3_ver29."):
        del sys.modules[name]
sys.path.insert(0, str(package_parent))
support_spec = importlib.util.spec_from_file_location("modal_support_" + uuid.uuid4().hex, support_file)
modal_support = importlib.util.module_from_spec(support_spec)
support_spec.loader.exec_module(modal_support)
from mission3_ver29.settings import default_settings
SETTINGS = default_settings()
SETTINGS.update(run_mode=RUN_MODE, source_run=SOURCE_RUN, resume_run=RESUME_RUN,
                input_zip=INPUT_ZIP, time_budget_minutes=TIME_BUDGET_MINUTES,
                invocation_started_monotonic=INVOCATION_STARTED,
                deadline_monotonic=DEADLINE_MONOTONIC)
SETTINGS, INVOCATION_DIR = modal_support.prepare(SETTINGS, VOLUME_DIR, package_parent)
environment = dict(os.environ)
environment["PYTHONPATH"] = str(package_parent) + os.pathsep + environment.get("PYTHONPATH", "")
environment["HF_HOME"] = SETTINGS["cache_dir"]
environment["HF_HUB_CACHE"] = str(Path(SETTINGS["cache_dir"]) / "hub")
environment["TOKENIZERS_PARALLELISM"] = "false"
environment["HF_HUB_DISABLE_TELEMETRY"] = "1"
environment["PYTHONUNBUFFERED"] = "1"
environment["PYTHONUTF8"] = "1"
modal_support.run_logged([sys.executable, "-u", "-c",
    "import torch; "
    "assert torch.cuda.is_available(), 'L4 GPU is not connected'; "
    "print('GPU:', torch.cuda.get_device_name(0)); "
    "print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda); "
    "print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))"],
    INVOCATION_DIR / "gpu_check.log", min(DEADLINE_MONOTONIC, time.monotonic() + 120),
    cwd=str(package_parent), env=environment)
PREPARED = True
print("입력 ZIP 검증 완료:", SETTINGS["input_zip"])
print("결과 저장 위치:", SETTINGS["output_dir"])
print("이번 호출 로그:", INVOCATION_DIR)


In [ ]:
# Preserve Modal's installed CUDA PyTorch and use uv or the existing pip.
from importlib.metadata import version
if not globals().get("PREPARED"):
    raise RuntimeError("앞선 코드 준비 셀의 오류부터 확인하세요.")
if globals().get("DEPENDENCIES_PREPARED"):
    raise RuntimeError("패키지 준비가 이미 끝났습니다. 다음 학습 셀을 실행하세요.")
torch_before = version("torch")
command = modal_support.install_command(package_dir / "requirements.txt",
                                         package_parent / "constraints.txt", torch_before)
modal_support.run_logged(command, INVOCATION_DIR / "install.log",
                         min(DEADLINE_MONOTONIC, time.monotonic() + 25 * 60),
                         cwd=str(package_parent), env=environment)
installed = {name: version(name) for name in ("torch", "transformers", "numpy", "scikit-learn")}
if installed["torch"] != torch_before:
    raise RuntimeError("CUDA PyTorch 버전이 바뀌었습니다. 학습을 시작하지 않습니다.")
modal_support.write_json(INVOCATION_DIR / "installed_packages.json", installed)
DEPENDENCIES_PREPARED = True
print("패키지 준비 완료:", installed)
print("남은 실행 시간:", round((DEADLINE_MONOTONIC - time.monotonic()) / 60, 1), "분")


In [ ]:
# Keep this cell running until the supervised child finishes and exports its results.
if not globals().get("PREPARED") or not globals().get("DEPENDENCIES_PREPARED"):
    raise RuntimeError("처음 셀부터 준비를 완료하세요.")
if globals().get("LAUNCHED"):
    raise RuntimeError("이 호출은 이미 시작했습니다. 새 실행은 설정 셀부터 시작하세요.")
LAUNCHED = True
RESULT = modal_support.execute(SETTINGS, INVOCATION_DIR, package_parent, environment,
                                DEADLINE_MONOTONIC)
print("이번 실행 시간:", round((time.monotonic() - INVOCATION_STARTED) / 60, 1), "분")


## 결과와 다음 실행

Files에서 Volume 아래 `exports/ver29_analysis_*.zip`을 내려받습니다.
작은 분석 ZIP에는 큰 가중치와 optimizer checkpoint가 없습니다.
`results/mission3/ver29/실행번호/` 전체 폴더를 삭제하지 마세요.
학습 로그는 별도의 `invocations/호출번호/training.log`에 저장됩니다.

- **`screening_complete`**: 후보 비교 완료. ZIP 결과를 검토한 뒤 full을 결정합니다.
- **`paused`**: 중간 저장 후 종료. 같은 L4와 `RUN_MODE`를 유지하고 `RESUME_RUN`에
  출력된 실행 폴더의 정확한 전체 경로를 넣습니다. `SOURCE_RUN`은 비웁니다.
- **`complete` + `submission_ready=true`**: full 결과. 공식 Validation 지표와 제출 파일을 확인합니다.
- **`failed` / `interrupted`**: 오류 또는 강제 종료. 완료 결과로 사용하지 않습니다.
  로그와 저장 상태를 먼저 확인합니다. 강제 종료가 정상적인 `paused`를 뜻하지는 않습니다.

**결과 검토 후 전체 학습**은 `RUN_MODE="full"`, `RESUME_RUN=""`로 바꾸고
`SOURCE_RUN`에 완료된 ver29 screening 폴더를 넣은 뒤 첫 셀부터 한 번 실행합니다.
선택된 1개·2개·4개 모델을 Training 전체 29,200개로 다시 학습합니다.
full이 paused로 끝나면 다음 실행은 `SOURCE_RUN=""`와 그 full 폴더의 `RESUME_RUN`을 사용합니다.

새 노트북에 옮길 때도 같은 Volume과 L4를 다시 연결하세요.
Volume 밖 `/tmp`에는 다시 만들 수 있는 코드와 압축 해제한 데이터만 둡니다.
입력 ZIP·모델 다운로드 캐시·학습 결과·로그는 Volume에 둡니다.
Modal은 Volume을 주기적으로 저장하고 정상 종료 때 최종 저장하지만 갑작스러운 종료는 별개입니다.
완료 출력과 저장된 파일을 확인한 뒤 **Stop kernel**로 GPU 사용을 끝내세요.

[Modal Notebooks 안내](https://modal.com/docs/guide/notebooks) ·
[Volume 저장 동작](https://modal.com/docs/guide/volumes#background-commits)
